# TinyGLASS — TG-384 inference efficiency

The default profile restores the already-trained TG-384 checkpoint: 384 × 384,
layer2 features, sigma 1, frozen ResNet-18, pose A and the fixed crop720.
Measure latency, throughput, parameters, checkpoint size, CUDA memory and GPU
energy with the same validation protocol used for the other model comparisons.
The saved architecture is restored strictly; no fit or calibration is performed.

Attach the extracted synthetic wheel dataset, select a Tesla T4 GPU, and run all
cells in a fresh Kaggle session. The default source is the verified Google Drive
run; enable Internet and the authorized secrets GDRIVE_CLIENT_ID,
GDRIVE_CLIENT_SECRET and GDRIVE_REFRESH_TOKEN. Alternatively set
CHECKPOINT_SOURCE="local" and MODEL_CHECKPOINT_PATH to the trusted final model.ckpt.
No pretrained weights, DTD textures or repository checkout are needed.

REFERENCE_PROFILE="TG-384", BENCHMARK_ENABLED=True and
TEST_EVALUATION_ENABLED=False are the defaults. Run only one profile per fresh
session so another model does not affect memory or device-wide energy results.
The configuration also retains the TG-512 profile (512 × 512, layer3, sigma 4).
To repeat its held-out test, select "TG-512", enable TEST_EVALUATION_ENABLED and
disable BENCHMARK_ENABLED. The TG-512 test remains a separate optional workflow.

Efficiency outputs use a new timestamped folder whose name includes input size,
feature layer and sigma under /kaggle/working/tinyglass_efficiency/.


In [ ]:
from __future__ import annotations

import copy
import csv
import hashlib
import importlib.util
import inspect
import json
import math
import os
import platform
import random
import subprocess
import sys
import threading
import time
import warnings
from collections import Counter
from collections.abc import Iterable, Mapping
from dataclasses import dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Sequence

import torch
from PIL import Image
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision.models import get_model, get_model_weights
from torchvision.models.feature_extraction import create_feature_extractor
from torchvision.transforms import InterpolationMode
from torchvision.transforms import functional as TF
from torchvision.transforms.functional import gaussian_blur, pil_to_tensor
from tqdm.auto import tqdm


In [ ]:
from __future__ import annotations

MODEL_NAME = "tinyglass"
# Select once, then run all cells in a fresh session. Input/layer/sigma/Drive IDs
# are coupled to prevent accidentally benchmarking a differently trained model.
REFERENCE_PROFILE = "TG-384"  # TG-384 | TG-512
REFERENCE_PROFILES = {
    "TG-384": {
        "input_size": (384, 384),
        "feature_grid_resolution": "layer2",
        "gaussian_sigma": 1.0,
        "accepted_saved_sigmas": (1.0,),
        "drive_run": {
            "model": "tinyglass",
            "folder_id": "1Xo5ELdDZdMKsfhKjuDT2HaQm4RqyTonb",
            "checkpoint_file_id": "1L45AfJ4ZeFceXfy2houemOVc_3Kv4Qpr",
            "run_manifest_file_id": "1aij5HWk2iCH254Pu7Dgp5q2u7TgK_cO_",
            "comment": (
                "Verified 384/layer2/sigma1 reference run, seed42, 20260831_112751; "
                "frozen ResNet18, LAS hole severity weights 70/25/5."
            ),
        },
    },
    "TG-512": {
        "input_size": (512, 512),
        "feature_grid_resolution": "layer3",
        "gaussian_sigma": 4.0,
        "accepted_saved_sigmas": (1.0, 4.0),
        "drive_run": {
            "model": "tinyglass",
            "folder_id": "1hWFs0qKm4xqGiT98ZH8XAJCDTl06u5iL",
            "checkpoint_file_id": "1gq7zyfAJZ5Vza2GjLBsIA91lVKU-b3LW",
            "run_manifest_file_id": "1W7fK-FX5bwxEY5W_OTNNIRYtIDsHSsgu",
            "comment": (
                "Existing 512/layer3 checkpoint; its folder name incorrectly says "
                "384/layer2. Apply the previously selected sigma4 after strict restore."
            ),
        },
    },
}
if REFERENCE_PROFILE not in REFERENCE_PROFILES:
    raise ValueError(f"Unknown TinyGLASS reference profile: {REFERENCE_PROFILE!r}")
reference_profile_config = REFERENCE_PROFILES[REFERENCE_PROFILE]

CHECKPOINT_SOURCE = "google_drive"  # google_drive | local
MODEL_CHECKPOINT_PATH = None  # e.g. "/kaggle/input/my-tg384-run/model.ckpt"
DRIVE_RUN_FOLDER_ID_OVERRIDE = None
FORCE_CHECKPOINT_DOWNLOAD = False
DRIVE_CHECKPOINT_CACHE_DIR = Path("/kaggle/working/tinyglass_efficiency_checkpoints")
SELECTED_DRIVE_RUN = dict(reference_profile_config["drive_run"])

KAGGLE_INPUT = Path("/kaggle/input")
DATASET_ROOT_OVERRIDE = None  # optional extracted synthetic dataset directory
OUTPUT_ROOT = Path("/kaggle/working/tinyglass_efficiency")
MODEL_INPUT_SIZE = reference_profile_config["input_size"]
EXPECTED_FEATURE_GRID_RESOLUTION = reference_profile_config["feature_grid_resolution"]
INFERENCE_GAUSSIAN_SIGMA = reference_profile_config["gaussian_sigma"]
EXPECTED_SAVED_GAUSSIAN_SIGMAS = reference_profile_config["accepted_saved_sigmas"]
CAMERA_POSES = ("A_overhead",)
POSE_CROPS = {"A_overhead": (90, 240, 720, 720)}

# Default: TG-384 validation efficiency. The optional test section is TG-512 only.
BENCHMARK_ENABLED = True
TEST_EVALUATION_ENABLED = False
if TEST_EVALUATION_ENABLED and REFERENCE_PROFILE != "TG-512":
    raise ValueError("The optional sigma-4 test requires REFERENCE_PROFILE='TG-512'")
TEST_SPLIT = "test"
TEST_BATCH_SIZE = 1
TEST_NUM_WORKERS = 0
TEST_PIN_MEMORY = False
TEST_HISTOGRAM_BINS = 2048
TEST_PRO_BINS = 256
TEST_PRO_MAX_FPR = 0.30
TEST_NUM_EXTREME_EXAMPLES = 3
TEST_GROUP_FIELDS = ("severity", "camera_pose", "lighting", "wear")
EXPECTED_TEST_CONDITION_COUNTS = {"clean": 404, "hole": 404}
EXPECTED_MANIFEST_SHA256 = "afc4d44cf8330b94c52f21a314b797abdc553d517348b11489ac15e32a3c06fb"
# Informational consistency comparison for the optional TG-512 test only.
REFERENCE_TEST_IMAGE_METRICS = {
    "image_auroc": 0.7733034751494952,
    "image_average_precision": 0.8112556446243319,
}
BENCHMARK_SPLIT = "validation"
BENCHMARK_BATCH_SIZE = 1
BENCHMARK_NUM_WORKERS = 0
BENCHMARK_PIN_MEMORY = False
BENCHMARK_WARMUP_RUNS = 10
BENCHMARK_TIMED_RUNS = 50
RANDOM_SEED = 42
REQUIRE_CUDA = True

if CHECKPOINT_SOURCE not in {"google_drive", "local"}:
    raise ValueError("CHECKPOINT_SOURCE must be google_drive or local")

# Separate sustained GPU energy measurement; same settings as EfficientAD.
ENERGY_ENABLED = True
ENERGY_METHOD = "auto"  # NVML total-energy counter, or measured-power integration
ENERGY_REPEATS = 3
ENERGY_DURATION_SECONDS = 30.0
ENERGY_WARMUP_SECONDS = 5.0
ENERGY_POWER_SAMPLE_INTERVAL_SECONDS = 0.1


In [ ]:
from __future__ import annotations

# Installation is needed only for authenticated Drive downloads.
if CHECKPOINT_SOURCE == "google_drive":
    drive_requirements = {
        "google.oauth2.credentials": "google-auth",
        "googleapiclient": "google-api-python-client",
    }
    missing_packages = []
    for module_name, package_name in drive_requirements.items():
        try:
            installed = importlib.util.find_spec(module_name) is not None
        except ModuleNotFoundError:
            installed = False
        if not installed:
            missing_packages.append(package_name)
    if missing_packages:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing_packages])


In [ ]:
from __future__ import annotations

import importlib.util
import subprocess
import sys

# Optional sensor dependency. Works in an older live session too.
# Installation/sensor failure never fabricates an energy estimate.
if globals().get("BENCHMARK_ENABLED", False) and globals().get("ENERGY_ENABLED", True) and importlib.util.find_spec("pynvml") is None:
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "nvidia-ml-py"])
    except Exception as error:
        print(f"WARNING: NVML bindings could not be installed; energy may be unavailable: {error}")



## Standalone definitions

Preprocessing, dataset, loader, TinyGLASS, metrics, diagnostic accumulators,
evaluation runner and efficiency helper are explicit copies of the authoritative
repository code. Training methods remain in the copied model class to preserve
synchronization; no execution cell invokes them. The evaluation copies omit only
Hydra construction and unused plotting/export helpers. All saved model tensors
are restored from the final checkpoint without downloading initial backbone weights.


In [ ]:
from __future__ import annotations

RGBTriplet = tuple[float, float, float]
ImageSize = tuple[int, int]
SigmaRange = tuple[float, float]
CropBox = tuple[int, int, int, int]
IMAGENET_MEAN: RGBTriplet = (0.485, 0.456, 0.406)
IMAGENET_STD: RGBTriplet = (0.229, 0.224, 0.225)


@dataclass(frozen=True)
class PreprocessingConfig:
    """Optional preprocessing and train-time image augmentation settings.

    ``resize`` follows the torchvision convention ``(height, width)``. Every
    scalar augmentation value is disabled when ``None`` and otherwise denotes
    its maximum intensity. Photometric transforms affect RGB only.
    """

    resize: ImageSize | None = None
    resize_shorter_side: int | None = None
    center_crop: ImageSize | None = None
    pose_crop_enabled: bool = False
    pose_crops: Mapping[str, CropBox | None] = field(default_factory=dict)
    normalize_mean: RGBTriplet | None = None
    normalize_std: RGBTriplet | None = None
    augmentations_enabled: bool = True
    brightness: float | None = None
    contrast: float | None = None
    gamma: float | None = None
    saturation: float | None = None
    sensor_noise: float | None = None
    gaussian_noise: float | None = None
    gaussian_blur: SigmaRange | None = None

    def __post_init__(self) -> None:
        if self.resize is not None and self.resize_shorter_side is not None:
            raise ValueError("resize and resize_shorter_side are mutually exclusive")
        if self.resize is not None:
            if len(self.resize) != 2 or any(value < 1 for value in self.resize):
                raise ValueError("resize must be a positive (height, width) pair")
        if self.resize_shorter_side is not None and self.resize_shorter_side < 1:
            raise ValueError("resize_shorter_side must be positive")
        if self.center_crop is not None:
            if len(self.center_crop) != 2 or any(value < 1 for value in self.center_crop):
                raise ValueError("center_crop must be a positive (height, width) pair")
        for pose, crop in self.pose_crops.items():
            if not pose:
                raise ValueError("pose crop names cannot be empty")
            if crop is None:
                continue
            if len(crop) != 4:
                raise ValueError("pose crops must be (top, left, height, width)")
            if crop[2] < 1 or crop[3] < 1:
                raise ValueError("pose crop height and width must be positive")
            if crop[2] != crop[3]:
                raise ValueError("pose crops must be square")

        if (self.normalize_mean is None) != (self.normalize_std is None):
            raise ValueError("normalize_mean and normalize_std must be set together")
        if self.normalize_mean is not None:
            if len(self.normalize_mean) != 3 or len(self.normalize_std or ()) != 3:
                raise ValueError("normalization mean and std must contain three RGB values")
            if any(value <= 0 for value in self.normalize_std or ()):
                raise ValueError("normalization std values must be positive")

        for name in (
            "brightness",
            "contrast",
            "gamma",
            "saturation",
            "sensor_noise",
            "gaussian_noise",
        ):
            value = getattr(self, name)
            if value is not None and value < 0:
                raise ValueError(f"{name} must be non-negative or None")

        for name in ("brightness", "contrast", "gamma", "saturation"):
            value = getattr(self, name)
            if value is not None and value >= 1:
                raise ValueError(f"{name} must be smaller than 1")

        if self.gaussian_blur is not None:
            if len(self.gaussian_blur) != 2:
                raise ValueError("gaussian_blur must be a (min_sigma, max_sigma) pair")
            minimum, maximum = self.gaussian_blur
            if minimum <= 0 or maximum < minimum:
                raise ValueError("gaussian_blur requires 0 < min_sigma <= max_sigma")


class WheelPreprocessor:
    """Apply aligned geometric transforms and RGB-only photometric transforms."""

    def __init__(self, config: PreprocessingConfig | None = None) -> None:
        self.config = config or PreprocessingConfig()

    @staticmethod
    def _symmetric_factor(maximum_delta: float) -> float:
        return 1.0 + (2.0 * torch.rand(1).item() - 1.0) * maximum_delta

    @staticmethod
    def _odd_kernel_size(maximum_sigma: float) -> int:
        radius = max(1, round(3.0 * maximum_sigma))
        return 2 * radius + 1

    def __call__(
        self,
        image: torch.Tensor,
        target_mask: torch.Tensor,
        anomaly_mask: torch.Tensor,
        *,
        camera_pose: str | None = None,
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        config = self.config

        if config.pose_crop_enabled:
            if camera_pose is None:
                raise ValueError(
                    "camera_pose is required when pose cropping is enabled"
                )
            if camera_pose not in config.pose_crops:
                raise ValueError(f"No pose crop configured for {camera_pose!r}")
            crop = config.pose_crops[camera_pose]
            if crop is not None:
                top, left, height, width = crop
                image_height, image_width = image.shape[-2:]
                if height > image_height or width > image_width:
                    raise ValueError(
                        f"Pose crop {crop!r} for {camera_pose!r} cannot fit "
                        f"inside the input image {(image_height, image_width)!r} "
                        "without padding"
                    )
                top = min(max(top, 0), image_height - height)
                left = min(max(left, 0), image_width - width)
                image = TF.crop(image, top, left, height, width)
                target_mask = TF.crop(target_mask, top, left, height, width)
                anomaly_mask = TF.crop(anomaly_mask, top, left, height, width)

        if config.resize is not None:
            image = TF.resize(
                image,
                config.resize,
                interpolation=InterpolationMode.BILINEAR,
                antialias=True,
            )
            target_mask = TF.resize(
                target_mask,
                config.resize,
                interpolation=InterpolationMode.NEAREST,
            )
            anomaly_mask = TF.resize(
                anomaly_mask,
                config.resize,
                interpolation=InterpolationMode.NEAREST,
            )
        elif config.resize_shorter_side is not None:
            image = TF.resize(
                image,
                config.resize_shorter_side,
                interpolation=InterpolationMode.BILINEAR,
                antialias=True,
            )
            target_mask = TF.resize(
                target_mask,
                config.resize_shorter_side,
                interpolation=InterpolationMode.NEAREST,
            )
            anomaly_mask = TF.resize(
                anomaly_mask,
                config.resize_shorter_side,
                interpolation=InterpolationMode.NEAREST,
            )

        if config.center_crop is not None:
            image = TF.center_crop(image, config.center_crop)
            target_mask = TF.center_crop(target_mask, config.center_crop)
            anomaly_mask = TF.center_crop(anomaly_mask, config.center_crop)

        # Every model-facing split uses one stable numerical contract.  Keeping
        # uint8 for evaluation while train-time augmentation returned float32
        # would either fail in the backbone or change the input scale by 255x.
        image = TF.convert_image_dtype(image, torch.float32)

        if config.augmentations_enabled:
            if config.brightness is not None:
                image = TF.adjust_brightness(image, self._symmetric_factor(config.brightness))
            if config.contrast is not None:
                image = TF.adjust_contrast(image, self._symmetric_factor(config.contrast))
            if config.gamma is not None:
                image = TF.adjust_gamma(image, self._symmetric_factor(config.gamma))
            if config.saturation is not None:
                image = TF.adjust_saturation(image, self._symmetric_factor(config.saturation))

            if config.gaussian_blur is not None:
                minimum, maximum = config.gaussian_blur
                sigma = minimum + torch.rand(1).item() * (maximum - minimum)
                kernel_size = self._odd_kernel_size(maximum)
                image = TF.gaussian_blur(image, kernel_size, sigma)

            if config.sensor_noise is not None and config.sensor_noise > 0:
                signal_scale = image.clamp(0.0, 1.0).sqrt()
                image = image + torch.randn_like(image) * signal_scale * config.sensor_noise
            if config.gaussian_noise is not None and config.gaussian_noise > 0:
                image = image + torch.randn_like(image) * config.gaussian_noise

        image = image.clamp(0.0, 1.0)
        if config.normalize_mean is not None:
            image = TF.normalize(
                image,
                mean=config.normalize_mean,
                std=config.normalize_std,
            )

        return image, target_mask, anomaly_mask

    def image_for_display(self, image: torch.Tensor) -> torch.Tensor:
        """Return a float RGB image in [0, 1], undoing normalization if needed."""
        if image.dtype == torch.uint8:
            return TF.convert_image_dtype(image, torch.float32)

        result = image.detach().clone()
        if self.config.normalize_mean is not None:
            mean = result.new_tensor(self.config.normalize_mean).view(3, 1, 1)
            std = result.new_tensor(self.config.normalize_std).view(3, 1, 1)
            result = result * std + mean
        return result.clamp(0.0, 1.0)


In [ ]:
from __future__ import annotations

REQUIRED_COLUMNS = {
    "image_id",
    "split",
    "condition",
    "image_path",
    "target_mask_path",
    "anomaly_mask_path",
    "pair_id",
    "camera_pose",
}
VALID_SPLITS = ("train", "validation", "test")
VALID_CONDITIONS = {"clean", "hole"}
ARTIFACT_PATH_FIELDS = ("image_path", "target_mask_path", "anomaly_mask_path")


class CuriosityWheelDataset(Dataset[dict[str, Any]]):
    """Load one split of the extracted Curiosity wheel Kaggle dataset."""

    def __init__(
        self,
        root: str | Path,
        split: str,
        preprocessing: WheelPreprocessor | None = None,
        camera_poses: Sequence[str] | None = None,
    ) -> None:
        self.root = Path(root).expanduser().resolve()
        self.split = split
        self.preprocessing = preprocessing or WheelPreprocessor()

        if split not in VALID_SPLITS:
            raise ValueError(f"Unsupported split {split!r}; expected one of {VALID_SPLITS}")
        if not self.root.is_dir():
            raise NotADirectoryError(f"Dataset root is not a directory: {self.root}")

        manifest_path = self.root / "samples.csv"
        if not manifest_path.is_file():
            raise FileNotFoundError(f"Dataset manifest is missing: {manifest_path}")

        with manifest_path.open(encoding="utf-8", newline="") as stream:
            reader = csv.DictReader(stream)
            columns = set(reader.fieldnames or [])
            missing = REQUIRED_COLUMNS - columns
            if missing:
                raise ValueError(f"samples.csv is missing required columns: {sorted(missing)}")
            rows = list(reader)

        self._validate_manifest_rows(rows)
        if isinstance(camera_poses, str):
            raise TypeError("camera_poses must be a sequence of pose names, not a string")
        if camera_poses is None:
            self.camera_poses = None
        else:
            self.camera_poses = tuple(dict.fromkeys(str(pose) for pose in camera_poses))
            if not self.camera_poses or any(not pose for pose in self.camera_poses):
                raise ValueError("camera_poses must contain at least one non-empty pose")
            available_poses = {row.get("camera_pose", "") for row in rows}
            unknown_poses = set(self.camera_poses) - available_poses
            if unknown_poses:
                raise ValueError(
                    f"Unknown camera poses: {sorted(unknown_poses)}; "
                    f"available poses: {sorted(available_poses)}"
                )

        self.rows = [
            row
            for row in rows
            if row["split"] == split
            and (
                self.camera_poses is None
                or row.get("camera_pose") in self.camera_poses
            )
        ]

        if not self.rows:
            raise ValueError(
                f"No samples found for split {split!r} and "
                f"camera_poses={self.camera_poses!r}"
            )

        for row in self.rows:
            for field in ARTIFACT_PATH_FIELDS:
                relative_path = row[field]
                if relative_path and not (self.root / relative_path).is_file():
                    raise FileNotFoundError(
                        f"Dataset artifact is missing: {self.root / relative_path}"
                    )

    @classmethod
    def _validate_manifest_rows(cls, rows: list[dict[str, str]]) -> None:
        """Validate the complete manifest before selecting one split."""
        if not rows:
            raise ValueError("samples.csv contains no samples")

        image_ids: set[str] = set()
        pairs: dict[str, list[dict[str, str]]] = {}
        for row in rows:
            image_id = row["image_id"]
            if not image_id:
                raise ValueError("samples.csv contains an empty image_id")
            if image_id in image_ids:
                raise ValueError(f"Duplicate image_id in samples.csv: {image_id}")
            image_ids.add(image_id)

            row_split = row["split"]
            if row_split not in VALID_SPLITS:
                raise ValueError(
                    f"Unsupported split {row_split!r} in sample {image_id}; "
                    f"expected one of {VALID_SPLITS}"
                )

            condition = row["condition"]
            if condition not in VALID_CONDITIONS:
                raise ValueError(f"Unsupported condition {condition!r} in sample {image_id}")
            if not row["image_path"] or not row["target_mask_path"]:
                raise ValueError(f"Sample has incomplete artifact paths: {image_id}")
            if condition == "hole" and not row["anomaly_mask_path"]:
                raise ValueError(f"Hole sample has no anomaly mask: {image_id}")
            if condition == "clean" and row["anomaly_mask_path"]:
                raise ValueError(f"Clean sample unexpectedly has an anomaly mask: {image_id}")
            for field in ARTIFACT_PATH_FIELDS:
                if row[field]:
                    cls._validate_relative_path(row[field])

            pair_id = row["pair_id"]
            if condition == "hole" and not pair_id:
                raise ValueError(f"Hole sample has no pair_id: {image_id}")
            if pair_id:
                pairs.setdefault(pair_id, []).append(row)

        for pair_id, pair_rows in pairs.items():
            conditions = {row["condition"] for row in pair_rows}
            splits = {row["split"] for row in pair_rows}
            if len(pair_rows) != 2 or conditions != VALID_CONDITIONS:
                raise ValueError(
                    f"Pair {pair_id!r} must contain exactly one clean and one hole sample"
                )
            if len(splits) != 1:
                raise ValueError(f"Pair {pair_id!r} crosses dataset splits: {sorted(splits)}")

    @staticmethod
    def _validate_relative_path(value: str) -> None:
        path = Path(value)
        if path.is_absolute() or ".." in path.parts:
            raise ValueError(f"Invalid dataset-relative path: {value!r}")

    def _load_image(self, relative_path: str, mode: str) -> Image.Image:
        path = self.root / relative_path
        if not path.is_file():
            raise FileNotFoundError(f"Dataset artifact is missing: {path}")
        with Image.open(path) as image:
            return image.convert(mode)

    def __len__(self) -> int:
        return len(self.rows)

    def __getitem__(self, index: int) -> dict[str, Any]:
        row = self.rows[index]
        image = pil_to_tensor(self._load_image(row["image_path"], "RGB"))
        target_mask = pil_to_tensor(self._load_image(row["target_mask_path"], "L"))

        if image.shape[1:] != target_mask.shape[1:]:
            raise ValueError(f"Image/target mask size mismatch for {row['image_id']}")

        anomaly_path = row["anomaly_mask_path"]
        if anomaly_path:
            anomaly_mask = pil_to_tensor(self._load_image(anomaly_path, "L"))
            if anomaly_mask.shape != target_mask.shape:
                raise ValueError(f"Target/anomaly mask size mismatch for {row['image_id']}")
        else:
            # Clean samples use an empty mask so every batch has one stable schema.
            anomaly_mask = torch.zeros_like(target_mask)

        image, target_mask, anomaly_mask = self.preprocessing(
            image,
            target_mask,
            anomaly_mask,
            camera_pose=row.get("camera_pose"),
        )

        return {
            "image": image,
            "target_mask": target_mask,
            "anomaly_mask": anomaly_mask,
            "label": torch.tensor(row["condition"] == "hole", dtype=torch.long),
            "has_anomaly_mask": bool(anomaly_path),
            "metadata": {
                key: value
                for key, value in row.items()
                if key not in {"image_path", "target_mask_path", "anomaly_mask_path"}
            },
        }


In [ ]:
from __future__ import annotations

# Kaggle exposes attached datasets as extracted directories.
def find_dataset_root(input_root: Path = KAGGLE_INPUT) -> Path:
    candidates = [
        manifest.parent
        for manifest in input_root.rglob("samples.csv")
        if (manifest.parent / "images").is_dir()
        and (manifest.parent / "masks").is_dir()
    ]
    if not candidates:
        raise FileNotFoundError(
            f"No valid dataset found under {input_root}: "
            "samples.csv, images/, and masks/ are required."
        )
    if len(candidates) > 1:
        raise RuntimeError(f"Multiple dataset candidates found: {candidates}")
    return candidates[0]

def build_dataloader(
    root: str | Path,
    split: str,
    *,
    batch_size: int = 4,
    num_workers: int = 0,
    pin_memory: bool = False,
    seed: int = 42,
    preprocessing: WheelPreprocessor | None = None,
    camera_poses: tuple[str, ...] | None = None,
) -> DataLoader:
    """Build one loader; only the training split is shuffled."""
    if batch_size < 1:
        raise ValueError("batch_size must be at least 1")
    if num_workers < 0:
        raise ValueError("num_workers cannot be negative")

    dataset = CuriosityWheelDataset(
        root,
        split,
        preprocessing=preprocessing,
        camera_poses=camera_poses,
    )
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=split == "train",
        num_workers=num_workers,
        pin_memory=pin_memory,
        persistent_workers=num_workers > 0,
        generator=torch.Generator().manual_seed(seed),
    )


In [ ]:
from __future__ import annotations

CHECKPOINT_SCHEMA_VERSION = 2


@dataclass(frozen=True)
class AnomalyPrediction:
    """Model-independent anomaly output with normalized scores."""

    anomaly_score: torch.Tensor
    anomaly_map: torch.Tensor

    def __post_init__(self) -> None:
        if self.anomaly_score.ndim != 1:
            raise ValueError("anomaly_score must have shape [B]")
        if self.anomaly_map.ndim != 4 or self.anomaly_map.shape[1] != 1:
            raise ValueError("anomaly_map must have shape [B, 1, H, W]")
        if self.anomaly_score.shape[0] != self.anomaly_map.shape[0]:
            raise ValueError("anomaly_score and anomaly_map batch sizes must match")
        for name, value in (
            ("anomaly_score", self.anomaly_score),
            ("anomaly_map", self.anomaly_map),
        ):
            if not value.is_floating_point() or not torch.isfinite(value).all():
                raise ValueError(f"{name} must contain finite floating-point values")
            if value.numel() and (value.min() < 0 or value.max() > 1):
                raise ValueError(f"{name} must be normalized to [0, 1]")


class AnomalyDetector(nn.Module):
    """Small common interface shared by all anomaly detectors."""

    @property
    def is_fitted(self) -> bool:
        raise NotImplementedError

    def fit(
        self,
        train_loader: Iterable[Mapping[str, Any]],
        *,
        device: torch.device | str,
        validation_loader: Iterable[Mapping[str, Any]] | None = None,
        work_dir: str | Path | None = None,
        resume: bool = False,
    ) -> AnomalyDetector:
        raise NotImplementedError

    @torch.no_grad()
    def predict(self, images: torch.Tensor) -> AnomalyPrediction:
        raise NotImplementedError

    def checkpoint_config(self) -> dict[str, Any]:
        """Return behavior-affecting constructor settings for strict reloads."""
        return {}

    def save(self, path: str | Path, *, metadata: Mapping[str, Any] | None = None) -> Path:
        """Save model state and small run metadata in one checkpoint."""
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        torch.save(
            {
                "schema_version": CHECKPOINT_SCHEMA_VERSION,
                "model_class": f"{type(self).__module__}.{type(self).__qualname__}",
                "model_config": self.checkpoint_config(),
                "model_state_dict": self.state_dict(),
                "metadata": dict(metadata or {}),
            },
            path,
        )
        return path

    def _prepare_state_dict_for_load(self, state_dict: Mapping[str, torch.Tensor]) -> None:
        """Allow models with dynamically sized buffers to prepare before load."""

    def load(
        self,
        path: str | Path,
        *,
        map_location: torch.device | str | None = None,
    ) -> dict[str, Any]:
        """Load a trusted project checkpoint and return its metadata."""
        payload = torch.load(Path(path), map_location=map_location)
        if not isinstance(payload, dict) or "model_state_dict" not in payload:
            raise ValueError("Checkpoint does not contain model_state_dict")
        if payload.get("schema_version") != CHECKPOINT_SCHEMA_VERSION:
            raise ValueError(
                f"Unsupported checkpoint schema {payload.get('schema_version')!r}; "
                f"expected {CHECKPOINT_SCHEMA_VERSION}"
            )
        expected_class = f"{type(self).__module__}.{type(self).__qualname__}"
        if payload.get("model_class") != expected_class:
            raise ValueError(
                f"Checkpoint model class {payload.get('model_class')!r} does not "
                f"match {expected_class!r}"
            )
        saved_config = payload.get("model_config")
        expected_config = self.checkpoint_config()
        if saved_config != expected_config:
            raise ValueError(
                "Checkpoint model configuration does not match the current model: "
                f"saved={saved_config!r}, current={expected_config!r}"
            )
        state_dict = payload["model_state_dict"]
        self._prepare_state_dict_for_load(state_dict)
        self.load_state_dict(state_dict)
        return dict(payload.get("metadata", {}))


In [ ]:
from __future__ import annotations

def build_explicit_torchvision_backbone(
    backbone: str, *, pretrained: bool, weights_name: str,
) -> nn.Module:
    weights = None
    if pretrained:
        weights_enum = get_model_weights(backbone)
        try:
            weights = getattr(weights_enum, weights_name)
        except AttributeError as error:
            raise ValueError(
                f"Unknown torchvision weights {weights_name!r} for {backbone!r}"
            ) from error
    return get_model(backbone, weights=weights)



def _rand_perlin_2d(
    shape: tuple[int, int], res: tuple[int, int], *, device: torch.device,
) -> torch.Tensor:
    """Torch port of the Perlin generator used by official SuperSimpleNet."""
    delta = (res[0] / shape[0], res[1] / shape[1])
    repeats = (shape[0] // res[0], shape[1] // res[1])
    y = torch.arange(0, res[0], delta[0], device=device)
    x = torch.arange(0, res[1], delta[1], device=device)
    grid = torch.stack(torch.meshgrid(y, x, indexing="ij"), dim=-1) % 1
    angles = 2 * math.pi * torch.rand(res[0] + 1, res[1] + 1, device=device)
    gradients = torch.stack((torch.cos(angles), torch.sin(angles)), dim=-1)

    def tiled(y_slice, x_slice):
        return (
            gradients[y_slice[0]:y_slice[1], x_slice[0]:x_slice[1]]
            .repeat_interleave(repeats[0], 0)
            .repeat_interleave(repeats[1], 1)
        )

    def dot(gradient, shift):
        offsets = torch.stack(
            (grid[:shape[0], :shape[1], 0] + shift[0],
             grid[:shape[0], :shape[1], 1] + shift[1]),
            dim=-1,
        )
        return (offsets * gradient[:shape[0], :shape[1]]).sum(dim=-1)

    n00 = dot(tiled((0, -1), (0, -1)), (0, 0))
    n10 = dot(tiled((1, None), (0, -1)), (-1, 0))
    n01 = dot(tiled((0, -1), (1, None)), (0, -1))
    n11 = dot(tiled((1, None), (1, None)), (-1, -1))
    fade = lambda value: 6 * value**5 - 15 * value**4 + 10 * value**3
    blend = fade(grid[:shape[0], :shape[1]])
    return math.sqrt(2) * torch.lerp(
        torch.lerp(n00, n10, blend[..., 0]),
        torch.lerp(n01, n11, blend[..., 0]),
        blend[..., 1],
    )



def _capture_training_rng_state(train_loader) -> dict[str, Any]:
    generator = getattr(train_loader, "generator", None)
    return {
        "torch_rng_state": torch.get_rng_state(),
        "python_rng_state": random.getstate(),
        "cuda_rng_state": (
            torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None
        ),
        "train_loader_generator_state": (
            generator.get_state() if generator is not None else None
        ),
    }


def _restore_training_rng_state(checkpoint: dict[str, Any], train_loader) -> None:
    generator_state = checkpoint.get("train_loader_generator_state")
    generator = getattr(train_loader, "generator", None)
    if generator_state is not None:
        if generator is None:
            raise TypeError("The train loader cannot restore its generator state")
        generator.set_state(
            torch.as_tensor(generator_state, dtype=torch.uint8, device="cpu")
        )
    torch.set_rng_state(
        torch.as_tensor(checkpoint["torch_rng_state"], dtype=torch.uint8, device="cpu")
    )
    if checkpoint.get("python_rng_state") is not None:
        random.setstate(checkpoint["python_rng_state"])
    cuda_rng_state = checkpoint.get("cuda_rng_state")
    if torch.cuda.is_available() and cuda_rng_state is not None:
        torch.cuda.set_rng_state_all([
            torch.as_tensor(state, dtype=torch.uint8, device="cpu")
            for state in cuda_rng_state
        ])


def _save_checkpoint_atomic(checkpoint: dict[str, Any], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    torch.save(checkpoint, temporary)
    temporary.replace(path)


def _clean_images(batch: Mapping[str, Any], device: torch.device) -> torch.Tensor:
    if "image" not in batch:
        raise KeyError("Every training batch must contain 'image'")
    if torch.any(torch.as_tensor(batch.get("label", 0)) != 0):
        raise ValueError("fit accepts only clean training images")
    return batch["image"].to(device, non_blocking=True)


def _image_auc(scores: list[float], labels: list[int]) -> float:
    positives = [score for score, label in zip(scores, labels) if label == 1]
    negatives = [score for score, label in zip(scores, labels) if label == 0]
    if not positives or not negatives:
        return float("nan")
    wins = sum(p > n for p in positives for n in negatives)
    ties = sum(p == n for p in positives for n in negatives)
    return (wins + 0.5 * ties) / (len(positives) * len(negatives))


In [ ]:
from __future__ import annotations

# Ported from the MIT-licensed official implementation:
# https://github.com/ETH-PBL/TinyGLASS


TINYGLASS_TRAINING_CHECKPOINT_FORMAT = "tinyglass_training_v4_configurable_las"


def _score_distribution(scores: torch.Tensor) -> dict[str, int | float | None]:
    scores = scores.detach().double().cpu().flatten()
    if scores.numel() == 0:
        return {
            "count": 0, "min": None, "max": None,
            "mean": None, "std": None,
            "greater_than_0_99_fraction": None,
            "greater_than_0_999_fraction": None,
        }
    return {
        "count": scores.numel(),
        "min": scores.min().item(),
        "max": scores.max().item(),
        "mean": scores.mean().item(),
        "std": scores.std(unbiased=False).item(),
        "greater_than_0_99_fraction": (scores > 0.99).double().mean().item(),
        "greater_than_0_999_fraction": (scores > 0.999).double().mean().item(),
    }


@torch.no_grad()
def _validation_diagnostics(
    model, loader, device: torch.device, max_batches: int | None,
) -> dict[str, Any]:
    if loader is None:
        raise ValueError(
            "TinyGLASS best-checkpoint selection requires validation_loader"
        )
    model.eval()
    score_chunks: list[torch.Tensor] = []
    label_chunks: list[torch.Tensor] = []
    for index, batch in enumerate(loader):
        if max_batches is not None and index >= max_batches:
            break
        prediction = model.predict(batch["image"].to(device, non_blocking=True))
        score_chunks.append(prediction.anomaly_score.detach().cpu())
        label_chunks.append(torch.as_tensor(batch["label"]).int().cpu())
    return _score_diagnostics(score_chunks, label_chunks)


def _score_diagnostics(
    score_chunks: list[torch.Tensor], label_chunks: list[torch.Tensor],
) -> dict[str, Any]:
    if not score_chunks:
        raise ValueError("validation_loader produced no images")
    scores = torch.cat(score_chunks).flatten()
    labels = torch.cat(label_chunks).flatten()
    if scores.shape != labels.shape:
        raise ValueError("Validation image scores and labels have different sizes")
    if not torch.isfinite(scores).all():
        raise ValueError("TinyGLASS validation scores must be finite")
    if not torch.all((labels == 0) | (labels == 1)):
        raise ValueError("TinyGLASS validation labels must be binary")
    if not torch.any(labels == 0) or not torch.any(labels == 1):
        raise ValueError("TinyGLASS validation requires clean and anomaly samples")
    return {
        "image_auroc": _image_auc(scores.tolist(), labels.tolist()),
        "raw_image_score_distribution": {
            "all": _score_distribution(scores),
            "clean": _score_distribution(scores[labels == 0]),
            "anomaly": _score_distribution(scores[labels == 1]),
        },
    }


@torch.no_grad()
def _build_validation_feature_cache(
    model, loader, device: torch.device, max_batches: int | None,
) -> list[tuple[torch.Tensor, torch.Tensor]]:
    if loader is None:
        raise ValueError(
            "TinyGLASS best-checkpoint selection requires validation_loader"
        )
    model.eval()
    cache: list[tuple[torch.Tensor, torch.Tensor]] = []
    for index, batch in enumerate(loader):
        if max_batches is not None and index >= max_batches:
            break
        features = model.features(
            batch["image"].to(device, non_blocking=True)
        ).detach().cpu()
        labels = torch.as_tensor(batch["label"]).int().cpu().flatten()
        if features.shape[0] != labels.numel():
            raise ValueError("Validation features and labels have different sizes")
        cache.append((features, labels))
    if not cache:
        raise ValueError("validation_loader produced no images")
    return cache


@torch.no_grad()
def _cached_validation_diagnostics(
    model, cache: list[tuple[torch.Tensor, torch.Tensor]], device: torch.device,
) -> dict[str, Any]:
    model.eval()
    score_chunks: list[torch.Tensor] = []
    label_chunks: list[torch.Tensor] = []
    for features, labels in cache:
        patch_scores = model.discriminator(
            features.to(device, non_blocking=True)
        )
        score_chunks.append(patch_scores.flatten(1).amax(1).cpu())
        label_chunks.append(labels)
    return _score_diagnostics(score_chunks, label_chunks)


class TinyGLASSFeatureExtractor(nn.Module):
    """TinyGLASS ResNet-18 layer2/layer3 patch-grid embedding."""

    def __init__(
        self, *, pretrained: bool, weights_name: str,
        patch_size: int = 3, output_channels_per_layer: int = 64,
        freeze_backbone: bool = True,
        feature_grid_resolution: str = "layer3",
    ) -> None:
        super().__init__()
        backbone = build_explicit_torchvision_backbone(
            "resnet18", pretrained=pretrained, weights_name=weights_name
        )
        self.extractor = create_feature_extractor(
            backbone, return_nodes={"layer2": "layer2", "layer3": "layer3"}
        )
        self.freeze_backbone = bool(freeze_backbone)
        self.extractor.requires_grad_(not self.freeze_backbone)
        self.patch_size = patch_size
        self.output_channels_per_layer = output_channels_per_layer
        if feature_grid_resolution not in {"layer2", "layer3"}:
            raise ValueError(
                "feature_grid_resolution must be either 'layer2' or 'layer3'"
            )
        self.feature_grid_resolution = feature_grid_resolution

    def train(self, mode: bool = True):
        super().train(False if self.freeze_backbone else mode)
        return self

    def _patch_grid(self, feature: torch.Tensor) -> torch.Tensor:
        patches = F.unfold(
            feature, kernel_size=self.patch_size, stride=1,
            padding=self.patch_size // 2,
        )
        return patches.reshape(
            feature.shape[0], -1, feature.shape[-2], feature.shape[-1]
        )

    def _reduce(self, feature: torch.Tensor) -> torch.Tensor:
        channels = feature.shape[1]
        if channels % self.output_channels_per_layer:
            raise ValueError(
                f"Patch channels {channels} are not divisible by "
                f"{self.output_channels_per_layer}"
            )
        return feature.reshape(
            feature.shape[0], self.output_channels_per_layer,
            channels // self.output_channels_per_layer,
            *feature.shape[-2:],
        ).mean(2)

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        if self.freeze_backbone:
            self.extractor.eval()
            with torch.no_grad():
                features = self.extractor(images)
        else:
            features = self.extractor(images)
        layer2 = self._reduce(self._patch_grid(features["layer2"]))
        layer3 = self._reduce(self._patch_grid(features["layer3"]))
        if self.feature_grid_resolution == "layer2":
            layer3 = F.interpolate(
                layer3, layer2.shape[-2:], mode="bilinear", align_corners=False
            )
        else:
            layer2 = F.adaptive_avg_pool2d(layer2, layer3.shape[-2:])
        return torch.cat((layer2, layer3), dim=1)


class TinyGLASSDiscriminator(nn.Module):
    def __init__(self, input_channels: int = 128, hidden_channels: int = 512) -> None:
        super().__init__()
        self.network = nn.Sequential(
            nn.Conv2d(input_channels, hidden_channels, 1),
            nn.BatchNorm2d(hidden_channels),
            nn.LeakyReLU(0.2),
            nn.Conv2d(hidden_channels, 1, 1, bias=False),
            nn.Sigmoid(),
        )
        self.apply(self._initialize)

    @staticmethod
    def _initialize(module: nn.Module) -> None:
        if isinstance(module, nn.Conv2d):
            nn.init.normal_(module.weight, 0.0, 0.02)
        elif isinstance(module, nn.BatchNorm2d):
            nn.init.normal_(module.weight, 1.0, 0.02)
            nn.init.zeros_(module.bias)

    def forward(self, features: torch.Tensor) -> torch.Tensor:
        return self.network(features).squeeze(1)


class DTDTextureSampler:
    """Texture loader and random augmentation used by TinyGLASS LAS."""

    _EXTENSIONS = {".jpg", ".jpeg", ".png"}

    def __init__(self, root: str | None, input_size: tuple[int, int]) -> None:
        self.root = None if root is None else Path(root)
        self.input_size = input_size
        self.paths = [] if self.root is None else sorted(
            path for path in self.root.rglob("*")
            if path.suffix.lower() in self._EXTENSIONS
        )

    @staticmethod
    def _random_operations(image: Image.Image) -> Image.Image:
        operations = [
            lambda value: TF.adjust_contrast(value, random.uniform(0.8, 1.2)),
            lambda value: TF.adjust_brightness(value, random.uniform(0.8, 1.2)),
            lambda value: TF.adjust_hue(
                TF.adjust_saturation(value, random.uniform(0.8, 1.2)),
                random.uniform(-0.2, 0.2),
            ),
            TF.hflip,
            TF.vflip,
            lambda value: TF.rgb_to_grayscale(value, num_output_channels=3),
            TF.autocontrast,
            TF.equalize,
            lambda value: TF.rotate(
                value, random.uniform(-45, 45),
                interpolation=InterpolationMode.BILINEAR,
            ),
        ]
        for index in random.sample(range(len(operations)), 3):
            image = operations[index](image)
        return image

    def sample(self, batch_size: int, *, device: torch.device) -> torch.Tensor:
        if not self.paths:
            raise RuntimeError("TinyGLASS requires DTD textures for LAS")
        values = []
        for index in torch.randint(len(self.paths), (batch_size,)).tolist():
            with Image.open(self.paths[index]) as image:
                image = TF.resize(
                    image.convert("RGB"), list(self.input_size),
                    interpolation=InterpolationMode.BILINEAR, antialias=True,
                )
                image = self._random_operations(image)
                tensor = pil_to_tensor(image).float().div_(255.0)
            mean = tensor.new_tensor((0.485, 0.456, 0.406)).view(3, 1, 1)
            std = tensor.new_tensor((0.229, 0.224, 0.225)).view(3, 1, 1)
            values.append((tensor - mean) / std)
        return torch.stack(values).to(device, non_blocking=True)


class TinyGLASSLAS(nn.Module):
    _MODES = {"texture", "hole", "mixed"}
    # Full-axis fractions relative to the equivalent wheel diameter. They are
    # derived from the Blender hole contract and its 0.485 m reference wheel.
    _AXIS_FRACTIONS = (
        ((0.027 / 0.485, 0.041 / 0.485), (0.010 / 0.485, 0.017 / 0.485)),
        ((0.044 / 0.485, 0.068 / 0.485), (0.015 / 0.485, 0.027 / 0.485)),
        ((0.073 / 0.485, 0.116 / 0.485), (0.022 / 0.485, 0.041 / 0.485)),
    )

    def __init__(
        self, *, input_size: tuple[int, int], blend_mean: float = 0.5,
        blend_std: float = 0.1, mode: str = "texture",
        hole_probability: float = 0.5,
        hole_luminance_range: tuple[float, float] = (0.18, 0.32),
        hole_severity_weights: tuple[float, float, float] = (0.5, 0.4, 0.1),
    ) -> None:
        super().__init__()
        if mode not in self._MODES:
            raise ValueError(f"LAS mode must be one of {sorted(self._MODES)}")
        if not 0 <= hole_probability <= 1:
            raise ValueError("LAS hole_probability must be in [0, 1]")
        if (
            len(hole_luminance_range) != 2
            or not 0 < hole_luminance_range[0] <= hole_luminance_range[1] < 1
        ):
            raise ValueError("LAS hole_luminance_range must satisfy 0 < min <= max < 1")
        if (
            len(hole_severity_weights) != 3
            or any(not math.isfinite(value) or value < 0 for value in hole_severity_weights)
            or sum(hole_severity_weights) <= 0
        ):
            raise ValueError(
                "LAS hole_severity_weights must contain three finite, "
                "non-negative values with a positive sum"
            )
        self.input_size = input_size
        self.blend_mean = blend_mean
        self.blend_std = blend_std
        self.mode = mode
        self.hole_probability = float(hole_probability)
        self.hole_luminance_range = tuple(float(v) for v in hole_luminance_range)
        severity_total = float(sum(hole_severity_weights))
        self.hole_severity_weights = tuple(
            float(value) / severity_total for value in hole_severity_weights
        )

    @property
    def requires_textures(self) -> bool:
        return self.mode in {"texture", "mixed"}

    @staticmethod
    def downsample_mask(
        image_mask: torch.Tensor, output_size: tuple[int, int],
    ) -> torch.Tensor:
        return F.adaptive_max_pool2d(image_mask, output_size=output_size)

    def _one_mask(
        self, device: torch.device, valid_mask: torch.Tensor | None = None,
    ) -> torch.Tensor:
        height, width = self.input_size
        if valid_mask is not None:
            if valid_mask.shape != (1, height, width):
                raise ValueError(
                    "TinyGLASS LAS target mask must have shape "
                    f"(1, {height}, {width})"
                )
            valid_mask = valid_mask.to(device=device).bool().squeeze(0)
            if not valid_mask.any():
                raise ValueError("TinyGLASS LAS target mask cannot be empty")
        for _ in range(32):
            masks = []
            for _ in range(2):
                scale_y = 2 ** int(torch.randint(0, 6, (1,)).item())
                scale_x = 2 ** int(torch.randint(0, 6, (1,)).item())
                noise = _rand_perlin_2d(
                    (height, width), (scale_y, scale_x), device=device
                )
                angle = float(torch.empty(()).uniform_(-90, 90))
                noise = TF.rotate(
                    noise[None], angle,
                    interpolation=InterpolationMode.BILINEAR,
                )[0]
                masks.append((noise > 0.5).float())
            choice = float(torch.rand(()))
            if choice > 2 / 3:
                mask = torch.clamp(masks[0] + masks[1], 0, 1)
            elif choice > 1 / 3:
                mask = masks[0] * masks[1]
            else:
                mask = masks[0]
            if valid_mask is not None:
                mask = mask * valid_mask
            if mask.any():
                return mask.unsqueeze(0)
        raise RuntimeError(
            "Could not generate a non-empty TinyGLASS Perlin mask inside the "
            "target wheel"
        )

    @staticmethod
    def _uniform(low: float, high: float) -> float:
        return float(torch.empty(()).uniform_(low, high))

    def _hole_contour(
        self, yy: torch.Tensor, xx: torch.Tensor, *, center_y: int,
        center_x: int, semi_long: float, semi_short: float,
    ) -> torch.Tensor:
        angle = self._uniform(-math.pi, math.pi)
        cos_angle, sin_angle = math.cos(angle), math.sin(angle)
        dx, dy = xx - center_x, yy - center_y
        major = (dx * cos_angle + dy * sin_angle) / semi_long
        minor = (-dx * sin_angle + dy * cos_angle) / semi_short
        radius = torch.sqrt(major.square() + minor.square())
        theta = torch.atan2(minor, major)
        profile = int(torch.multinomial(
            torch.tensor((0.45, 0.40, 0.15), device=yy.device), 1
        ).item())
        if profile == 0:  # jagged slit
            frequencies, amplitudes = (3, 5, 7), (0.11, 0.07, 0.04)
        elif profile == 1:  # branched tear: one pronounced, smooth side lobe
            frequencies, amplitudes = (2, 4, 6), (0.09, 0.06, 0.03)
        else:  # peeled window
            frequencies, amplitudes = (2, 3), (0.07, 0.04)
        boundary = torch.ones_like(radius)
        for frequency, amplitude in zip(frequencies, amplitudes):
            phase = self._uniform(-math.pi, math.pi)
            boundary = boundary + amplitude * torch.sin(frequency * theta + phase)
        if profile == 1:
            branch_angle = self._uniform(-math.pi, math.pi)
            angular_distance = torch.atan2(
                torch.sin(theta - branch_angle), torch.cos(theta - branch_angle)
            )
            boundary = boundary + 0.18 * torch.exp(
                -angular_distance.square() / (2 * 0.28 ** 2)
            )
        return radius <= boundary.clamp_min(0.65)

    def _one_hole_mask(
        self, device: torch.device, valid_mask: torch.Tensor,
    ) -> torch.Tensor:
        height, width = self.input_size
        if valid_mask.shape != (1, height, width):
            raise ValueError(
                "TinyGLASS hole LAS target mask must have shape "
                f"(1, {height}, {width})"
            )
        valid = valid_mask.to(device=device).bool().squeeze(0)
        locations = valid.nonzero(as_tuple=False)
        if locations.numel() == 0:
            raise ValueError("TinyGLASS hole LAS target mask cannot be empty")
        wheel_diameter = 2.0 * math.sqrt(float(valid.sum()) / math.pi)
        yy, xx = torch.meshgrid(
            torch.arange(height, device=device, dtype=torch.float32),
            torch.arange(width, device=device, dtype=torch.float32),
            indexing="ij",
        )
        # Re-sample size and centre together: perspective and partial visibility
        # can make the larger physical severity ranges impossible to contain.
        for _ in range(96):
            severity = int(torch.multinomial(
                torch.tensor(self.hole_severity_weights, device=device), 1
            ).item())
            long_range, short_range = self._AXIS_FRACTIONS[severity]
            long_axis = max(4.0, wheel_diameter * self._uniform(*long_range))
            short_axis = max(3.0, wheel_diameter * self._uniform(*short_range))
            ratio = long_axis / short_axis
            if not 2.0 <= ratio <= 4.5:
                continue
            center = locations[int(torch.randint(len(locations), (1,), device=device))]
            mask = self._hole_contour(
                yy, xx, center_y=int(center[0]), center_x=int(center[1]),
                semi_long=long_axis / 2, semi_short=short_axis / 2,
            )
            if mask.sum() >= 6 and not (mask & ~valid).any():
                return mask.float().unsqueeze(0)
        raise RuntimeError(
            "Could not place a connected hole-like LAS anomaly fully inside "
            "the target wheel"
        )

    def _hole_composite(
        self, image: torch.Tensor, mask: torch.Tensor,
    ) -> torch.Tensor:
        mean = image.new_tensor((0.485, 0.456, 0.406)).view(3, 1, 1)
        std = image.new_tensor((0.229, 0.224, 0.225)).view(3, 1, 1)
        rgb = (image * std + mean).clamp(0, 1)
        mask_4d = mask.unsqueeze(0)
        interior = 1 - F.max_pool2d(1 - mask_4d, 3, stride=1, padding=1)
        interior = interior.squeeze(0)
        rim = (mask - interior).clamp(0, 1)
        noise = torch.rand((1, *self.input_size), device=image.device)
        noise = F.avg_pool2d(noise.unsqueeze(0), 9, stride=1, padding=4).squeeze(0)
        noise = 0.85 + 0.30 * noise
        luminance = self._uniform(*self.hole_luminance_range)
        cavity = (rgb * luminance * noise).clamp(0, 1)
        rim_factor = self._uniform(0.45, 0.70)
        altered = cavity * interior + (rgb * rim_factor) * rim
        altered_normalized = (altered - mean) / std
        return image * (1 - mask) + altered_normalized * mask

    def forward(
        self, images: torch.Tensor, textures: torch.Tensor | None = None,
        target_masks: torch.Tensor | None = None,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        if target_masks is not None:
            expected_shape = (images.shape[0], 1, *self.input_size)
            if target_masks.shape != expected_shape:
                raise ValueError(
                    "TinyGLASS LAS target masks must match the image batch: "
                    f"expected {expected_shape}, got {tuple(target_masks.shape)}"
                )
        if self.requires_textures and textures is None:
            raise ValueError(f"TinyGLASS LAS mode '{self.mode}' requires textures")
        augmented_images, masks = [], []
        for index in range(images.shape[0]):
            use_hole = self.mode == "hole" or (
                self.mode == "mixed" and float(torch.rand(())) < self.hole_probability
            )
            valid_mask = None if target_masks is None else target_masks[index]
            if use_hole:
                if valid_mask is None:
                    raise ValueError("TinyGLASS hole LAS requires target masks")
                mask = self._one_hole_mask(images.device, valid_mask)
                augmented = self._hole_composite(images[index], mask)
            else:
                mask = self._one_mask(images.device, valid_mask)
                beta = float(torch.normal(
                    self.blend_mean, self.blend_std, size=(), device=images.device
                ).clamp(0.2, 0.8))
                augmented = images[index] * (1 - mask) + (
                    (1 - beta) * textures[index] + beta * images[index]
                ) * mask
            augmented_images.append(augmented)
            masks.append(mask)
        return torch.stack(augmented_images), torch.stack(masks)


def _tinyglass_focal_loss(
    probabilities: torch.Tensor, targets: torch.Tensor,
    gamma: float = 2.0,
) -> torch.Tensor:
    probabilities = probabilities.clamp(1e-5, 1 - 1e-5)
    targets = targets.float()
    pt = probabilities * targets + (1 - probabilities) * (1 - targets)
    return (-(1 - pt).pow(gamma) * torch.log(pt)).mean()


class TinyGLASSDeployment(nn.Module):
    def __init__(
        self, features: TinyGLASSFeatureExtractor,
        discriminator: TinyGLASSDiscriminator,
    ) -> None:
        super().__init__()
        self.features = copy.deepcopy(features)
        self.discriminator = copy.deepcopy(discriminator)

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        return self.discriminator(self.features(images)).unsqueeze(1)


class TinyGLASS(AnomalyDetector):
    """TinyGLASS with dedicated LAS, GAS, patch embedding and discriminator."""

    def __init__(
        self, *, pretrained: bool = True,
        weights_name: str = "IMAGENET1K_V1",
        input_size: tuple[int, int] = (256, 256), patch_size: int = 3,
        epochs: int = 640, learning_rate: float = 1e-4,
        weight_decay: float = 1e-2, noise_std: float = 0.015,
        radius_quantile: float = 0.75, hard_mining_quantile: float = 0.5,
        gas_steps: int = 20, gas_step_size: float = 0.001,
        hypersphere_projection: bool = True,
        feature_grid_resolution: str = "layer3",
        max_samples_per_epoch: int | None = 392,
        texture_root: str | None = None,
        require_texture_dataset: bool = True,
        blend_mean: float = 0.5, blend_std: float = 0.1,
        gaussian_sigma: float = 4.0,
        las_restrict_to_target_mask: bool = False,
        las_mode: str = "texture", las_hole_probability: float = 0.5,
        las_hole_luminance_range: tuple[float, float] = (0.18, 0.32),
        las_hole_severity_weights: tuple[float, float, float] = (0.5, 0.4, 0.1),
        freeze_backbone: bool = True,
        backbone_learning_rate: float = 1e-5,
        fixed_training_duration: bool = False,
        validation_interval: int = 1, validation_batches: int | None = None,
        cache_validation_features: bool = True,
        early_stopping_patience: int | None = 20,
        early_stopping_min_delta: float = 0.0,
        checkpoint_interval: int = 1,
    ) -> None:
        super().__init__()
        if not 0 < radius_quantile < 1:
            raise ValueError("radius_quantile must be in (0, 1)")
        if not 0 <= hard_mining_quantile < 1:
            raise ValueError("hard_mining_quantile must be in [0, 1)")
        self.pretrained = pretrained
        self.weights_name = weights_name
        self.input_size = tuple(input_size)
        self.patch_size = patch_size
        self.epochs = epochs
        self.learning_rate = learning_rate
        self.weight_decay = weight_decay
        self.noise_std = noise_std
        self.radius_quantile = radius_quantile
        self.hard_mining_quantile = hard_mining_quantile
        self.gas_steps = gas_steps
        self.gas_step_size = gas_step_size
        self.hypersphere_projection = hypersphere_projection
        if feature_grid_resolution not in {"layer2", "layer3"}:
            raise ValueError(
                "feature_grid_resolution must be either 'layer2' or 'layer3'"
            )
        self.feature_grid_resolution = feature_grid_resolution
        self.max_samples_per_epoch = max_samples_per_epoch
        self.require_texture_dataset = require_texture_dataset
        self.blend_mean = blend_mean
        self.blend_std = blend_std
        self.gaussian_sigma = gaussian_sigma
        self.las_restrict_to_target_mask = bool(las_restrict_to_target_mask)
        self.las_mode = str(las_mode)
        self.las_hole_probability = float(las_hole_probability)
        self.las_hole_luminance_range = tuple(las_hole_luminance_range)
        self.las_hole_severity_weights = tuple(las_hole_severity_weights)
        if self.las_mode in {"hole", "mixed"} and not self.las_restrict_to_target_mask:
            raise ValueError(
                "TinyGLASS hole/mixed LAS requires las_restrict_to_target_mask=true"
            )
        self.freeze_backbone = bool(freeze_backbone)
        if backbone_learning_rate <= 0:
            raise ValueError("backbone_learning_rate must be positive")
        self.backbone_learning_rate = float(backbone_learning_rate)
        self.fixed_training_duration = fixed_training_duration
        if validation_interval < 1:
            raise ValueError("validation_interval must be at least 1")
        self.validation_interval = validation_interval
        if validation_batches is not None and validation_batches < 1:
            raise ValueError("validation_batches must be positive or None")
        self.validation_batches = validation_batches
        self.cache_validation_features = bool(cache_validation_features)
        if not self.freeze_backbone and self.cache_validation_features:
            warnings.warn(
                "TinyGLASS validation feature cache disabled because "
                "freeze_backbone=false",
                UserWarning,
                stacklevel=2,
            )
            self.cache_validation_features = False
        if early_stopping_patience is not None and early_stopping_patience < 1:
            raise ValueError("early_stopping_patience must be positive or None")
        if early_stopping_min_delta < 0:
            raise ValueError("early_stopping_min_delta cannot be negative")
        self.early_stopping_patience = early_stopping_patience
        self.early_stopping_min_delta = early_stopping_min_delta
        if checkpoint_interval < 1:
            raise ValueError("checkpoint_interval must be at least 1")
        self.checkpoint_interval = checkpoint_interval

        self.features = TinyGLASSFeatureExtractor(
            pretrained=pretrained, weights_name=weights_name,
            patch_size=patch_size, freeze_backbone=self.freeze_backbone,
            feature_grid_resolution=self.feature_grid_resolution,
        )
        self.discriminator = TinyGLASSDiscriminator()
        self.textures = DTDTextureSampler(texture_root, self.input_size)
        self.las = TinyGLASSLAS(
            input_size=self.input_size,
            blend_mean=blend_mean, blend_std=blend_std,
            mode=self.las_mode,
            hole_probability=self.las_hole_probability,
            hole_luminance_range=self.las_hole_luminance_range,
            hole_severity_weights=self.las_hole_severity_weights,
        )
        self.las_hole_severity_weights = self.las.hole_severity_weights
        self.register_buffer("center", torch.zeros(128))
        self.register_buffer("fitted", torch.tensor(False))
        self.fit_summary: dict[str, Any] = {}

    @property
    def is_fitted(self) -> bool:
        return bool(self.fitted)

    def train(self, mode: bool = True):
        super().train(mode)
        if self.freeze_backbone:
            self.features.eval()
        return self

    def checkpoint_config(self) -> dict[str, Any]:
        return {
            "backbone": "resnet18", "pretrained": self.pretrained,
            "weights_name": self.weights_name, "input_size": list(self.input_size),
            "patch_size": self.patch_size, "target_embedding_dimension": 128,
            "epochs": self.epochs, "learning_rate": self.learning_rate,
            "weight_decay": self.weight_decay, "noise_std": self.noise_std,
            "radius_quantile": self.radius_quantile,
            "hard_mining_quantile": self.hard_mining_quantile,
            "gas_steps": self.gas_steps, "gas_step_size": self.gas_step_size,
            "hypersphere_projection": self.hypersphere_projection,
            "feature_grid_resolution": self.feature_grid_resolution,
            "max_samples_per_epoch": self.max_samples_per_epoch,
            "require_texture_dataset": self.require_texture_dataset,
            "blend_mean": self.blend_mean, "blend_std": self.blend_std,
            "gaussian_sigma": self.gaussian_sigma,
            "las_restrict_to_target_mask": self.las_restrict_to_target_mask,
            "las_mode": self.las_mode,
            "las_hole_probability": self.las_hole_probability,
            "las_hole_luminance_range": list(self.las_hole_luminance_range),
            "las_hole_severity_weights": list(self.las_hole_severity_weights),
            "training_algorithm": TINYGLASS_TRAINING_CHECKPOINT_FORMAT,
            "freeze_backbone": self.freeze_backbone,
            "backbone_learning_rate": self.backbone_learning_rate,
            "fixed_training_duration": self.fixed_training_duration,
            "validation_interval": self.validation_interval,
            "validation_batches": self.validation_batches,
            "cache_validation_features": self.cache_validation_features,
            "early_stopping_patience": self.early_stopping_patience,
            "early_stopping_min_delta": self.early_stopping_min_delta,
        }

    def _training_state(self) -> dict[str, Any]:
        return {
            "features": self.features.state_dict(),
            "discriminator": self.discriminator.state_dict(),
            "center": self.center.detach().clone(),
        }

    def _load_training_state(self, state: dict[str, Any]) -> None:
        self.features.load_state_dict(state["features"])
        self.discriminator.load_state_dict(state["discriminator"])
        self.center.copy_(state["center"].to(self.center.device))

    @torch.no_grad()
    def _compute_center(self, train_loader, device: torch.device) -> int:
        total = torch.zeros_like(self.center, device=device)
        patches = 0
        features_were_training = self.features.training
        self.features.eval()
        try:
            for batch in tqdm(
                train_loader, desc="TinyGLASS feature center",
                unit="batch", leave=False,
            ):
                features = self.features(_clean_images(batch, device))
                flattened = features.permute(0, 2, 3, 1).reshape(-1, 128)
                total += flattened.sum(0)
                patches += flattened.shape[0]
        finally:
            self.features.train(features_were_training)
        if patches == 0:
            raise ValueError("train_loader produced no images")
        self.center.copy_(total / patches)
        return patches

    def _project_gas(
        self, gas: torch.Tensor, true: torch.Tensor,
        center: torch.Tensor, radius: torch.Tensor,
    ) -> torch.Tensor:
        gas_flat = gas.permute(0, 2, 3, 1).reshape(-1, 128)
        true_flat = true.permute(0, 2, 3, 1).reshape(-1, 128)
        base = center if self.hypersphere_projection else true_flat
        lower = radius if self.hypersphere_projection else gas_flat.new_tensor(0.5)
        vector = gas_flat - base
        norm = torch.norm(vector, dim=1).clamp_min(1e-10)
        alpha = torch.clamp(norm, lower, 2 * lower)
        projected = base + vector * (alpha / norm).unsqueeze(1)
        return projected.reshape_as(gas.permute(0, 2, 3, 1)).permute(0, 3, 1, 2)

    def _training_loss(
        self, true_features: torch.Tensor, fake_features: torch.Tensor,
        feature_mask: torch.Tensor,
    ) -> torch.Tensor | None:
        batch, channels, height, width = true_features.shape
        true_flat = true_features.permute(0, 2, 3, 1).reshape(-1, channels)
        fake_flat = fake_features.permute(0, 2, 3, 1).reshape(-1, channels)
        mask_flat = feature_mask.reshape(-1).bool()
        center = self.center.reshape(1, -1).expand_as(true_flat)
        true_points = torch.cat((fake_flat[~mask_flat], true_flat), dim=0)
        true_centers = torch.cat((center[~mask_flat], center), dim=0)
        radius = torch.quantile(
            torch.norm(true_points - true_centers, dim=1), self.radius_quantile
        ).detach().clamp_min(1e-6)

        gas = (
            true_features + torch.normal(
                0, self.noise_std, size=true_features.shape,
                device=true_features.device,
            )
        ).detach().requires_grad_(True)
        bce_loss = None
        for gas_step in range(self.gas_steps + 1):
            joint_scores = self.discriminator(torch.cat((true_features, gas), dim=0))
            true_scores, gas_scores = joint_scores.split(batch, dim=0)
            bce_loss = (
                F.binary_cross_entropy(true_scores, torch.zeros_like(true_scores))
                + F.binary_cross_entropy(gas_scores, torch.ones_like(gas_scores))
            )
            if gas_step == self.gas_steps:
                break
            gradient = torch.autograd.grad(
                F.binary_cross_entropy(gas_scores, torch.ones_like(gas_scores)),
                gas,
            )[0]
            gradient = gradient / torch.norm(
                gradient, dim=1, keepdim=True
            ).clamp_min(1e-10)
            gas = (gas + self.gas_step_size * gradient).detach()
            if (gas_step + 1) % 5 == 0:
                gas = self._project_gas(gas, true_features, center, radius)
            gas.requires_grad_(True)

        if not mask_flat.any():
            return None
        if self.hypersphere_projection:
            anomalous = fake_flat[mask_flat]
            anomaly_center = center[mask_flat]
            vector = anomalous - anomaly_center
            norm = torch.norm(vector, dim=1).clamp_min(1e-10)
            alpha = torch.clamp(norm, 2 * radius, 4 * radius)
            fake_flat = fake_flat.clone()
            fake_flat[mask_flat] = anomaly_center + vector * (alpha / norm).unsqueeze(1)
            fake_features = fake_flat.reshape(
                batch, height, width, channels
            ).permute(0, 3, 1, 2)

        fake_scores = self.discriminator(fake_features)
        distance = (fake_scores - feature_mask.squeeze(1)).square()
        if self.hard_mining_quantile > 0:
            threshold = torch.quantile(distance.detach(), self.hard_mining_quantile)
            selected = distance >= threshold
            fake_scores = fake_scores[selected]
            target = feature_mask.squeeze(1)[selected]
        else:
            target = feature_mask.squeeze(1)
        return bce_loss + _tinyglass_focal_loss(fake_scores, target)

    def fit(
        self, train_loader, *, device, validation_loader=None,
        work_dir=None, resume=False,
    ) -> TinyGLASS:
        device = torch.device(device)
        if not self.fixed_training_duration and validation_loader is None:
            raise ValueError(
                "TinyGLASS best-checkpoint selection requires validation_loader"
            )
        self.to(device)
        if self.freeze_backbone:
            optimizer = torch.optim.AdamW(
                self.discriminator.parameters(), lr=self.learning_rate * 2,
                weight_decay=self.weight_decay,
            )
        else:
            optimizer = torch.optim.AdamW(
                [
                    {
                        "params": self.discriminator.parameters(),
                        "lr": self.learning_rate * 2,
                    },
                    {
                        "params": self.features.parameters(),
                        "lr": self.backbone_learning_rate,
                    },
                ],
                weight_decay=self.weight_decay,
            )
        best_auc = -math.inf
        best_state = None
        best_epoch = None
        validation_history: list[dict[str, Any]] = []
        no_improvement_validations = 0
        stopped_early = False
        steps = 0
        skipped_empty_masks = 0
        center_patches = 0
        center_patches_total = 0
        center_recomputations = 0
        start_epoch = 0
        completed_epoch = 0
        validation_cache_images = 0
        validation_cache_bytes = 0
        validation_feature_cache = None
        checkpoint_path = (
            None if work_dir is None else Path(work_dir) / "tinyglass_training.ckpt"
        )
        if resume:
            if checkpoint_path is None or not checkpoint_path.is_file():
                raise FileNotFoundError(
                    "resume=true requires work_dir/tinyglass_training.ckpt"
                )
            checkpoint = torch.load(
                checkpoint_path, map_location="cpu", weights_only=False
            )
            if checkpoint.get("format") != TINYGLASS_TRAINING_CHECKPOINT_FORMAT:
                raise ValueError("Unsupported TinyGLASS training checkpoint")
            saved_config = checkpoint.get("model_config")
            if (
                not isinstance(saved_config, dict)
                or saved_config.get("freeze_backbone") != self.freeze_backbone
            ):
                raise ValueError(
                    "TinyGLASS freeze_backbone mismatch; start a new run with "
                    "resume=false"
                )
            if saved_config != self.checkpoint_config():
                raise ValueError("TinyGLASS training checkpoint config mismatch")
            self._load_training_state(checkpoint["model"])
            optimizer.load_state_dict(checkpoint["optimizer"])
            start_epoch = int(checkpoint["epoch"])
            completed_epoch = start_epoch
            if not 0 <= start_epoch <= self.epochs:
                raise ValueError("Invalid epoch in TinyGLASS training checkpoint")
            steps = int(checkpoint.get("steps", 0))
            skipped_empty_masks = int(checkpoint.get("skipped_empty_masks", 0))
            center_patches = int(checkpoint.get("center_patches", 0))
            center_patches_total = int(
                checkpoint.get("center_patches_total", center_patches)
            )
            center_recomputations = int(
                checkpoint.get("center_recomputations", int(center_patches > 0))
            )
            best_auc = float(checkpoint.get("best_auc", -math.inf))
            best_state = checkpoint.get("best_state")
            best_epoch = checkpoint.get("best_epoch")
            validation_history = list(checkpoint.get("validation_history", []))
            no_improvement_validations = int(
                checkpoint.get("no_improvement_validations", 0)
            )
            stopped_early = bool(checkpoint.get("stopped_early", False))
            validation_cache_images = int(
                checkpoint.get("validation_cache_images", 0)
            )
            validation_cache_bytes = int(
                checkpoint.get("validation_cache_bytes", 0)
            )
            _restore_training_rng_state(checkpoint, train_loader)
        else:
            if self.las.requires_textures and not self.textures.paths:
                raise FileNotFoundError(
                    "TinyGLASS training requires texture_root=dtd/images"
                )
            if self.freeze_backbone:
                center_patches = self._compute_center(train_loader, device)
                center_patches_total += center_patches
                center_recomputations += 1
        if (
            self.las.requires_textures
            and not stopped_early
            and start_epoch < self.epochs
            and not self.textures.paths
        ):
            raise FileNotFoundError(
                "TinyGLASS resume requires the original DTD texture dataset"
            )
        if (
            not stopped_early
            and start_epoch < self.epochs
            and not self.fixed_training_duration
            and self.cache_validation_features
        ):
            validation_feature_cache = _build_validation_feature_cache(
                self, validation_loader, device, self.validation_batches
            )
            validation_cache_images = sum(
                labels.numel() for _, labels in validation_feature_cache
            )
            validation_cache_bytes = sum(
                features.numel() * features.element_size()
                for features, _ in validation_feature_cache
            )
        remaining_epochs = (
            () if stopped_early else range(start_epoch + 1, self.epochs + 1)
        )
        epoch_progress = tqdm(
            remaining_epochs,
            initial=start_epoch, total=self.epochs, desc="TinyGLASS training",
            unit="epoch", dynamic_ncols=True,
        )
        for epoch in epoch_progress:
            completed_epoch = epoch
            if not self.freeze_backbone:
                center_patches = self._compute_center(train_loader, device)
                center_patches_total += center_patches
                center_recomputations += 1
            self.train(True)
            samples = 0
            produced = False
            for batch in train_loader:
                produced = True
                images = _clean_images(batch, device)
                textures = (
                    self.textures.sample(images.shape[0], device=device)
                    if self.las.requires_textures else None
                )
                target_masks = None
                if self.las_restrict_to_target_mask:
                    if "target_mask" not in batch:
                        raise KeyError(
                            "las_restrict_to_target_mask=true requires target_mask "
                            "in every training batch"
                        )
                    target_masks = batch["target_mask"].to(
                        device, non_blocking=True
                    )
                augmented, image_mask = self.las(
                    images, textures, target_masks=target_masks
                )
                if self.freeze_backbone:
                    with torch.no_grad():
                        true_features = self.features(images)
                        fake_features = self.features(augmented)
                else:
                    true_features = self.features(images)
                    fake_features = self.features(augmented)
                feature_mask = self.las.downsample_mask(
                    image_mask, output_size=true_features.shape[-2:]
                )
                loss = self._training_loss(
                    true_features.detach() if self.freeze_backbone else true_features,
                    fake_features.detach() if self.freeze_backbone else fake_features,
                    feature_mask,
                )
                if loss is None:
                    skipped_empty_masks += 1
                    continue
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()
                epoch_progress.set_postfix(
                    loss=f"{loss.detach().item():.4f}", refresh=False
                )
                steps += 1
                samples += images.shape[0]
                if self.max_samples_per_epoch and samples >= self.max_samples_per_epoch:
                    break
            if not produced:
                raise ValueError("train_loader produced no images")
            if (
                not self.fixed_training_duration
                and (epoch % self.validation_interval == 0 or epoch == self.epochs)
            ):
                self.fitted.fill_(True)
                diagnostics = (
                    _cached_validation_diagnostics(
                        self, validation_feature_cache, device
                    )
                    if validation_feature_cache is not None
                    else _validation_diagnostics(
                        self, validation_loader, device, self.validation_batches
                    )
                )
                diagnostics = {"epoch": epoch, **diagnostics}
                validation_history.append(diagnostics)
                auc = diagnostics["image_auroc"]
                saturation = diagnostics["raw_image_score_distribution"]["all"]
                epoch_progress.set_postfix(
                    val_auc=f"{auc:.4f}",
                    score_gt_0999=(
                        f"{saturation['greater_than_0_999_fraction']:.3f}"
                    ),
                    refresh=False,
                )
                if auc > best_auc + self.early_stopping_min_delta:
                    best_auc = auc
                    best_epoch = epoch
                    best_state = copy.deepcopy(self._training_state())
                    no_improvement_validations = 0
                else:
                    no_improvement_validations += 1
                if (
                    self.early_stopping_patience is not None
                    and no_improvement_validations >= self.early_stopping_patience
                ):
                    stopped_early = True
            if (
                checkpoint_path is not None
                and (
                    epoch % self.checkpoint_interval == 0
                    or epoch == self.epochs
                    or stopped_early
                )
            ):
                _save_checkpoint_atomic({
                    "format": TINYGLASS_TRAINING_CHECKPOINT_FORMAT,
                    "model_config": self.checkpoint_config(),
                    "model": self._training_state(),
                    "optimizer": optimizer.state_dict(),
                    "epoch": epoch,
                    "steps": steps,
                    "skipped_empty_masks": skipped_empty_masks,
                    "center_patches": center_patches,
                    "center_patches_total": center_patches_total,
                    "center_recomputations": center_recomputations,
                    "best_auc": best_auc,
                    "best_state": best_state,
                    "best_epoch": best_epoch,
                    "validation_history": validation_history,
                    "no_improvement_validations": no_improvement_validations,
                    "stopped_early": stopped_early,
                    "validation_cache_images": validation_cache_images,
                    "validation_cache_bytes": validation_cache_bytes,
                    **_capture_training_rng_state(train_loader),
                }, checkpoint_path)
            if stopped_early:
                break
        if best_state is not None:
            self._load_training_state(best_state)
        self.fitted.fill_(True)
        self.eval()
        self.fit_summary = {
            "epochs": self.epochs, "steps": steps,
            "epochs_completed": completed_epoch,
            "center_patches": center_patches,
            "center_patches_total": center_patches_total,
            "center_recomputations": center_recomputations,
            "skipped_empty_masks": skipped_empty_masks,
            "las_restrict_to_target_mask": self.las_restrict_to_target_mask,
            "las_mode": self.las_mode,
            "las_requires_textures": self.las.requires_textures,
            "las_hole_severity_weights": list(self.las_hole_severity_weights),
            "fixed_training_duration": self.fixed_training_duration,
            "freeze_backbone": self.freeze_backbone,
            "backbone_learning_rate": self.backbone_learning_rate,
            "selected_validation_image_auroc": (
                None if best_state is None else best_auc
            ),
            "selected_epoch": best_epoch,
            "stopped_early": stopped_early,
            "no_improvement_validations": no_improvement_validations,
            "validation_feature_cache_enabled": (
                not self.fixed_training_duration and self.cache_validation_features
            ),
            "validation_cache_images": validation_cache_images,
            "validation_cache_bytes": validation_cache_bytes,
            "validation_history": validation_history,
        }
        if checkpoint_path is not None:
            _save_checkpoint_atomic({
                "format": TINYGLASS_TRAINING_CHECKPOINT_FORMAT,
                "model_config": self.checkpoint_config(),
                "model": self._training_state(),
                "optimizer": optimizer.state_dict(),
                "epoch": completed_epoch,
                "steps": steps,
                "skipped_empty_masks": skipped_empty_masks,
                "center_patches": center_patches,
                "center_patches_total": center_patches_total,
                "center_recomputations": center_recomputations,
                "best_auc": best_auc,
                "best_state": best_state,
                "best_epoch": best_epoch,
                "validation_history": validation_history,
                "no_improvement_validations": no_improvement_validations,
                "stopped_early": stopped_early,
                "validation_cache_images": validation_cache_images,
                "validation_cache_bytes": validation_cache_bytes,
                **_capture_training_rng_state(train_loader),
            }, checkpoint_path)
        return self

    @torch.no_grad()
    def _patch_scores(self, images: torch.Tensor) -> torch.Tensor:
        return self.discriminator(self.features(images)).unsqueeze(1)

    @torch.no_grad()
    def predict_with_raw(self, images: torch.Tensor):
        if not self.is_fitted:
            raise RuntimeError("TinyGLASS must be fitted before predict")
        self.eval()
        patch_scores = self._patch_scores(images)
        anomaly_map = F.interpolate(
            patch_scores, images.shape[-2:], mode="bilinear", align_corners=False
        )
        if self.gaussian_sigma > 0:
            kernel = 2 * math.ceil(3 * self.gaussian_sigma) + 1
            anomaly_map = gaussian_blur(
                anomaly_map, [kernel, kernel],
                [self.gaussian_sigma, self.gaussian_sigma],
            )
        anomaly_score = patch_scores.flatten(1).amax(1)
        prediction = AnomalyPrediction(
            anomaly_score.clamp(0, 1), anomaly_map.clamp(0, 1)
        )
        return prediction, anomaly_score, patch_scores

    @torch.no_grad()
    def predict(self, images: torch.Tensor) -> AnomalyPrediction:
        return self.predict_with_raw(images)[0]

    def to_deployment_module(self) -> nn.Module:
        if not self.is_fitted:
            raise RuntimeError("TinyGLASS must be fitted before export")
        return TinyGLASSDeployment(self.features, self.discriminator).eval()


In [ ]:
from __future__ import annotations

# Authenticated Google Drive download with atomic write and size verification.
DRIVE_SCOPE = "https://www.googleapis.com/auth/drive.file"
DRIVE_FOLDER_MIME = "application/vnd.google-apps.folder"


def get_kaggle_secret(name: str, *, required: bool = False) -> str | None:
    try:
        from kaggle_secrets import UserSecretsClient
        value = UserSecretsClient().get_secret(name)
    except Exception:
        value = None
    if required and not value:
        raise RuntimeError(
            f"Missing required Kaggle secret {name!r}. Add the same Google Drive "
            "credentials used by the roadmap notebook."
        )
    return value


def build_drive_service():
    from google.oauth2.credentials import Credentials
    from googleapiclient.discovery import build

    credentials = Credentials(
        token=None,
        refresh_token=get_kaggle_secret("GDRIVE_REFRESH_TOKEN", required=True),
        token_uri="https://oauth2.googleapis.com/token",
        client_id=get_kaggle_secret("GDRIVE_CLIENT_ID", required=True),
        client_secret=get_kaggle_secret("GDRIVE_CLIENT_SECRET", required=True),
        scopes=[DRIVE_SCOPE],
    )
    return build("drive", "v3", credentials=credentials, cache_discovery=False)


def parse_drive_item_id(value: str) -> str:
    from urllib.parse import parse_qs, urlparse

    value = str(value or "").strip()
    if not value:
        raise ValueError("Google Drive ID/URL cannot be empty")
    if "/" not in value and "?" not in value:
        item_id = value
    else:
        parsed = urlparse(value)
        parts = [part for part in parsed.path.split("/") if part]
        item_id = parse_qs(parsed.query).get("id", [None])[0]
        if item_id is None:
            for marker in ("d", "folders"):
                if marker in parts and parts.index(marker) + 1 < len(parts):
                    item_id = parts[parts.index(marker) + 1]
                    break
    if not item_id or not all(
        character.isalnum() or character in "-_" for character in item_id
    ):
        raise ValueError(f"Invalid Google Drive ID/URL: {value!r}")
    return item_id


def drive_children(service, parent_id: str, name: str | None = None) -> list[dict]:
    backslash = chr(92)
    escaped_parent = parent_id.replace("'", backslash + "'")
    query = [f"'{escaped_parent}' in parents", "trashed = false"]
    if name is not None:
        escaped_name = name.replace(backslash, backslash * 2).replace(
            "'", backslash + "'"
        )
        query.append(f"name = '{escaped_name}'")
    return service.files().list(
        q=" and ".join(query),
        spaces="drive",
        fields="files(id,name,mimeType,size,parents)",
        pageSize=1000,
    ).execute(num_retries=3).get("files", [])


def selected_drive_run() -> dict[str, Any]:
    if DRIVE_RUN_FOLDER_ID_OVERRIDE:
        return {
            "model": MODEL_NAME,
            "folder_id": parse_drive_item_id(DRIVE_RUN_FOLDER_ID_OVERRIDE),
            "checkpoint_file_id": None,
            "compatible_with_current_preset": None,
            "comment": "Manual override; strict checkpoint configuration still applies.",
        }
    if SELECTED_DRIVE_RUN is None:
        raise RuntimeError(
            f"No verified Drive checkpoint is registered for {MODEL_NAME}. "
            "Use CHECKPOINT_SOURCE='local' and set MODEL_CHECKPOINT_PATH."
        )
    return dict(SELECTED_DRIVE_RUN)


def resolve_drive_checkpoint_file(service, run: Mapping[str, Any]) -> dict[str, Any]:
    folder_id = parse_drive_item_id(run["folder_id"])
    matches = [
        item
        for item in drive_children(service, folder_id, "model.ckpt")
        if item["mimeType"] != DRIVE_FOLDER_MIME
    ]
    if len(matches) != 1:
        raise RuntimeError(
            f"Expected exactly one model.ckpt in Drive folder {folder_id}, "
            f"found {len(matches)}"
        )
    item = matches[0]
    expected_file_id = run.get("checkpoint_file_id")
    if expected_file_id and item["id"] != expected_file_id:
        raise ValueError(
            "Drive folder checkpoint ID changed: "
            f"expected {expected_file_id}, found {item['id']}"
        )
    return item


def download_drive_checkpoint(
    service,
    run: Mapping[str, Any],
) -> tuple[Path, dict[str, Any]]:
    from googleapiclient.http import MediaIoBaseDownload

    item = resolve_drive_checkpoint_file(service, run)
    file_id = item["id"]
    destination = DRIVE_CHECKPOINT_CACHE_DIR / MODEL_NAME / f"{file_id}.ckpt"
    destination.parent.mkdir(parents=True, exist_ok=True)
    expected_size = int(item.get("size") or 0)
    if (
        not FORCE_CHECKPOINT_DOWNLOAD
        and destination.is_file()
        and expected_size
        and destination.stat().st_size == expected_size
    ):
        print(f"Reusing size-verified cached checkpoint: {destination}")
        return destination, {
            **item,
            "cached": True,
            "run_folder_id": parse_drive_item_id(run["folder_id"]),
        }

    temporary = destination.with_suffix(destination.suffix + ".part")
    temporary.unlink(missing_ok=True)
    request = service.files().get_media(fileId=file_id)
    try:
        with temporary.open("wb") as handle, tqdm(
            total=expected_size or None,
            desc=f"Downloading {MODEL_NAME} model.ckpt",
            unit="B",
            unit_scale=True,
            leave=False,
            dynamic_ncols=True,
        ) as progress:
            downloader = MediaIoBaseDownload(
                handle,
                request,
                chunksize=8 * 1024 * 1024,
            )
            downloaded = 0
            done = False
            while not done:
                status, done = downloader.next_chunk(num_retries=5)
                if status is not None:
                    current = int(status.resumable_progress)
                    progress.update(max(0, current - downloaded))
                    downloaded = current
        if expected_size and temporary.stat().st_size != expected_size:
            raise IOError(
                f"Incomplete Drive checkpoint: {temporary.stat().st_size} / "
                f"{expected_size} bytes"
            )
        os.replace(temporary, destination)
    except Exception:
        temporary.unlink(missing_ok=True)
        raise
    return destination, {
        **item,
        "cached": False,
        "run_folder_id": parse_drive_item_id(run["folder_id"]),
    }


def download_selected_checkpoint_from_drive() -> tuple[Path, dict[str, Any]]:
    run = selected_drive_run()
    if run.get("model") != MODEL_NAME:
        raise ValueError(
            f"Drive run model {run.get('model')!r} does not match {MODEL_NAME!r}"
        )
    service = build_drive_service()
    checkpoint_path, metadata = download_drive_checkpoint(service, run)
    print(
        f"Drive run folder: {parse_drive_item_id(run['folder_id'])} | "
        f"checkpoint file: {metadata['id']} | bytes={metadata.get('size', 'unknown')}"
    )
    return checkpoint_path, {"run": run, "file": metadata}


In [ ]:
from __future__ import annotations

def load_frozen_tinyglass(
    checkpoint_path: str | Path, *, device: torch.device | str,
) -> tuple[TinyGLASS, dict[str, Any]]:
    """Restore the saved configuration exactly, then apply the selected sigma."""
    checkpoint_path = Path(checkpoint_path).expanduser()
    if not checkpoint_path.is_file():
        raise FileNotFoundError(f"Final model checkpoint is missing: {checkpoint_path}")
    # Only trusted project checkpoints are supported; no unsafe pickle fallback.
    payload = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
    required = {"schema_version", "model_class", "model_config", "model_state_dict"}
    if not isinstance(payload, dict) or not required <= set(payload):
        raise ValueError("Expected a final model.ckpt, not a training checkpoint")
    if payload["schema_version"] != CHECKPOINT_SCHEMA_VERSION:
        raise ValueError("Unsupported final checkpoint schema")
    if str(payload["model_class"]).rsplit(".", 1)[-1] != "TinyGLASS":
        raise ValueError("The supplied checkpoint is not TinyGLASS")
    if not isinstance(payload["model_config"], dict):
        raise ValueError("Checkpoint model_config must be a dictionary")
    saved_config = dict(payload["model_config"])

    if tuple(saved_config.get("input_size") or ()) != tuple(MODEL_INPUT_SIZE):
        raise ValueError(
            f"{REFERENCE_PROFILE} requires input_size={MODEL_INPUT_SIZE}, "
            f"found {saved_config.get('input_size')}"
        )
    if saved_config.get("feature_grid_resolution") != EXPECTED_FEATURE_GRID_RESOLUTION:
        raise ValueError(
            f"{REFERENCE_PROFILE} requires {EXPECTED_FEATURE_GRID_RESOLUTION} features, "
            f"found {saved_config.get('feature_grid_resolution')!r}"
        )
    if saved_config.get("freeze_backbone") is not True:
        raise ValueError("The selected TinyGLASS checkpoint must have a frozen backbone")
    if saved_config.get("las_mode") != "hole":
        raise ValueError("The selected checkpoint must use LAS hole")
    if tuple(saved_config.get("las_hole_severity_weights") or ()) != (0.70, 0.25, 0.05):
        raise ValueError("The selected checkpoint must use LAS hole severity weights 70/25/5")
    if float(saved_config.get("gaussian_sigma", -1)) not in EXPECTED_SAVED_GAUSSIAN_SIGMAS:
        raise ValueError(
            f"{REFERENCE_PROFILE} requires saved sigma in {EXPECTED_SAVED_GAUSSIAN_SIGMAS}, "
            f"found {saved_config.get('gaussian_sigma')}"
        )

    metadata = dict(payload.get("metadata") or {})
    preprocessing = dict(metadata.get("preprocessing") or {})
    if preprocessing:
        expected_preprocessing = {
            "input_size": list(MODEL_INPUT_SIZE),
            "resize": list(MODEL_INPUT_SIZE),
            "resize_shorter_side": None,
            "center_crop": None,
            "pose_crop_enabled": True,
            "normalize_mean": list(IMAGENET_MEAN),
            "normalize_std": list(IMAGENET_STD),
        }
        for key, expected in expected_preprocessing.items():
            if key in preprocessing and preprocessing[key] != expected:
                raise ValueError(f"Checkpoint preprocessing mismatch: {key}")
        saved_pose_crop = (preprocessing.get("pose_crops") or {}).get("A_overhead")
        if saved_pose_crop is not None and tuple(saved_pose_crop) != tuple(POSE_CROPS["A_overhead"]):
            raise ValueError("Checkpoint pose-A crop does not match the fixed crop720")
    dataset_metadata = dict(metadata.get("dataset") or {})
    if dataset_metadata.get("camera_poses") is not None:
        if tuple(dataset_metadata["camera_poses"]) != tuple(CAMERA_POSES):
            raise ValueError("The selected checkpoint must have been trained on pose A only")

    # Constructor settings come from the checkpoint, not from roadmap defaults.
    constructor_fields = set(inspect.signature(TinyGLASS.__init__).parameters) - {"self"}
    config_only_fields = {"backbone", "target_embedding_dimension", "training_algorithm"}
    unknown_fields = set(saved_config) - constructor_fields - config_only_fields
    if unknown_fields:
        raise ValueError(f"Unrecognized TinyGLASS checkpoint fields: {sorted(unknown_fields)}")
    constructor_config = {
        key: value for key, value in saved_config.items() if key in constructor_fields
    }
    saved_pretrained = constructor_config["pretrained"]
    constructor_config["pretrained"] = False
    constructor_config["texture_root"] = None
    restored = TinyGLASS(**constructor_config)
    restored.pretrained = saved_pretrained
    if restored.checkpoint_config() != saved_config:
        raise ValueError("Saved TinyGLASS configuration does not match the current implementation exactly")
    restored._prepare_state_dict_for_load(payload["model_state_dict"])
    restored.load_state_dict(payload["model_state_dict"], strict=True)
    if not restored.is_fitted:
        raise ValueError("The supplied final checkpoint contains an unfitted model")
    restored.fit_summary = dict(metadata.get("fit_summary") or {})
    # Keep TG-384 sigma1; the TG-512 profile applies its previously selected sigma4.
    restored.gaussian_sigma = float(INFERENCE_GAUSSIAN_SIGMA)
    restored.to(device).eval()
    restore_info = {
        "reference_profile": REFERENCE_PROFILE,
        "checkpoint_sha256": sha256_file(checkpoint_path),
        "checkpoint_path": str(checkpoint_path),
        "checkpoint_size_bytes": checkpoint_path.stat().st_size,
        "checkpoint_model_class": payload["model_class"],
        "source_run_id": metadata.get("run_id"),
        "saved_model_config": saved_config,
        "inference_model_config": restored.checkpoint_config(),
        "saved_gaussian_sigma": float(saved_config["gaussian_sigma"]),
        "inference_gaussian_sigma": float(restored.gaussian_sigma),
        "sigma_override_after_strict_restore": (
            float(saved_config["gaussian_sigma"]) != float(restored.gaussian_sigma)
        ),
        "source_manifest_sha256": dataset_metadata.get("manifest_sha256"),
        "training_performed": False,
        "accuracy_evaluation_performed": False,
        "sigma_selection_performed": False,
    }
    return restored, restore_info


def sha256_file(path: str | Path) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def save_json_atomic(value: Any, path: str | Path) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False) + "\n", encoding="utf-8")
    temporary.replace(path)
    return path


In [ ]:
from __future__ import annotations

import threading

def _latency_summary(milliseconds: list[float]) -> dict[str, float]:
    if not milliseconds:
        raise ValueError("Latency samples cannot be empty")
    ordered = sorted(float(value) for value in milliseconds)

    def percentile(fraction: float) -> float:
        position = fraction * (len(ordered) - 1)
        lower, upper = math.floor(position), math.ceil(position)
        weight = position - lower
        return ordered[lower] * (1 - weight) + ordered[upper] * weight

    return {
        "minimum_ms": ordered[0],
        "mean_ms": sum(ordered) / len(ordered),
        "median_ms": percentile(0.50),
        "p90_ms": percentile(0.90),
        "p95_ms": percentile(0.95),
        "maximum_ms": ordered[-1],
    }


def _cuda_memory_baseline(device: torch.device) -> tuple[int, int] | None:
    if device.type != "cuda":
        return None
    torch.cuda.synchronize(device)
    torch.cuda.empty_cache()
    baseline = (
        torch.cuda.memory_allocated(device), torch.cuda.memory_reserved(device)
    )
    torch.cuda.reset_peak_memory_stats(device)
    return baseline


def _cuda_memory_report(
    device: torch.device, baseline: tuple[int, int] | None,
) -> dict[str, float] | None:
    if baseline is None:
        return None
    allocated, reserved = baseline
    peak_allocated = torch.cuda.max_memory_allocated(device)
    peak_reserved = torch.cuda.max_memory_reserved(device)
    return {
        "baseline_allocated_mib": allocated / 2**20,
        "baseline_reserved_mib": reserved / 2**20,
        "peak_allocated_mib": peak_allocated / 2**20,
        "peak_reserved_mib": peak_reserved / 2**20,
        "incremental_peak_allocated_mib": (peak_allocated - allocated) / 2**20,
        "incremental_peak_reserved_mib": (peak_reserved - reserved) / 2**20,
    }


@torch.inference_mode()
def benchmark_anomaly_inference(
    detector: Any,
    loader: Any,
    *,
    device: torch.device | str,
    warmup_runs: int = 10,
    timed_runs: int = 50,
    split: str = "validation",
) -> dict[str, Any]:
    """Measure predict-only and sequential batch-one pipeline latency.

    ``predict_only`` reuses one preprocessed, device-resident input. The other
    measurements include the loader's complete work, which for the synthetic
    dataset includes annotation I/O and aligned mask transforms. They are not
    a mask-free deployment benchmark. CUDA allocator peaks include all live
    tensors on the selected device, not just the model, and exclude driver
    allocations. No power sensor or energy estimate is used.
    """
    if getattr(loader, "batch_size", None) != 1:
        raise ValueError("Inference benchmark loader must use batch_size=1")
    if getattr(loader, "num_workers", 0) != 0:
        raise ValueError("Sequential inference benchmark requires num_workers=0")
    if not isinstance(warmup_runs, int) or not isinstance(timed_runs, int):
        raise TypeError("warmup_runs and timed_runs must be integers")
    if warmup_runs < 1 or timed_runs < 1:
        raise ValueError("warmup_runs and timed_runs must be positive")
    if not detector.is_fitted:
        raise RuntimeError("Fit or restore the detector before benchmarking")

    device = torch.device(device)
    uses_cuda = device.type == "cuda"
    was_training = detector.training
    detector.to(device)
    detector.eval()

    def synchronize() -> None:
        if uses_cuda:
            torch.cuda.synchronize(device)

    def next_batch(iterator):
        try:
            return next(iterator), iterator
        except StopIteration:
            iterator = iter(loader)
            try:
                return next(iterator), iterator
            except StopIteration as error:
                raise ValueError("Inference benchmark loader is empty") from error

    try:
        iterator = iter(loader)
        batch, iterator = next_batch(iterator)
        images = batch["image"].to(device, non_blocking=uses_cuda)
        if images.ndim != 4 or images.shape[0] != 1:
            raise ValueError("Inference benchmark requires one BCHW image")
        input_shape, input_dtype = list(images.shape), str(images.dtype)
        for _ in range(warmup_runs):
            prediction = detector.predict(images)
            del prediction
        synchronize()
        baseline = _cuda_memory_baseline(device)
        predict_ms = []
        for _ in range(timed_runs):
            synchronize()
            start = time.perf_counter()
            prediction = detector.predict(images)
            synchronize()
            predict_ms.append((time.perf_counter() - start) * 1_000)
            # Do not retain the previous output across the next forward.
            del prediction
        predict_memory = _cuda_memory_report(device, baseline)
        del images, batch, iterator

        iterator = iter(loader)
        for _ in range(warmup_runs):
            batch, iterator = next_batch(iterator)
            images = batch["image"].to(device, non_blocking=uses_cuda)
            prediction = detector.predict(images)
            synchronize()
            del prediction, images, batch
        baseline = _cuda_memory_baseline(device)
        preprocessing_ms, accelerator_ms, pipeline_ms = [], [], []
        iterator = iter(loader)
        for _ in range(timed_runs):
            synchronize()
            start = time.perf_counter()
            batch, iterator = next_batch(iterator)
            preprocessing_end = time.perf_counter()
            images = batch["image"].to(device, non_blocking=uses_cuda)
            if list(images.shape) != input_shape or str(images.dtype) != input_dtype:
                raise ValueError("Benchmark inputs must have one stable shape and dtype")
            prediction = detector.predict(images)
            synchronize()
            end = time.perf_counter()
            preprocessing_ms.append((preprocessing_end - start) * 1_000)
            accelerator_ms.append((end - preprocessing_end) * 1_000)
            pipeline_ms.append((end - start) * 1_000)
            del prediction, images, batch
        pipeline_memory = _cuda_memory_report(device, baseline)

        predict_summary = _latency_summary(predict_ms)
        accelerator_summary = _latency_summary(accelerator_ms)
        pipeline_summary = _latency_summary(pipeline_ms)
        parameters = list(detector.parameters())
        buffers = list(detector.buffers())
        bank = getattr(detector, "memory_bank", None)
        return {
            "schema_version": 1,
            "scope": (
                "predict_only: one device-resident preprocessed image; pipeline: "
                "sequential loader (including annotation I/O and mask transforms), "
                "host-to-device transfer and detector.predict; no worker prefetch"
            ),
            "memory_scope": (
                "CUDA allocator totals include the model, memory bank and other live "
                "tensors on this device; driver allocations are excluded"
            ),
            "split": split,
            "batch_size": 1,
            "warmup_runs": warmup_runs,
            "timed_runs": timed_runs,
            "device": str(device),
            "gpu_name": torch.cuda.get_device_name(device) if uses_cuda else None,
            "input_shape": input_shape,
            "input_dtype": input_dtype,
            "model": type(detector).__name__,
            "gaussian_sigma": getattr(detector, "gaussian_sigma", None),
            "total_parameters": sum(parameter.numel() for parameter in parameters),
            "trainable_parameters": sum(
                parameter.numel() for parameter in parameters if parameter.requires_grad
            ),
            "parameters_mib": sum(
                parameter.numel() * parameter.element_size() for parameter in parameters
            ) / 2**20,
            "buffers_mib": sum(
                buffer.numel() * buffer.element_size() for buffer in buffers
            ) / 2**20,
            "memory_bank_shape": list(bank.shape) if bank is not None else None,
            "memory_bank_mib": (
                bank.numel() * bank.element_size() / 2**20 if bank is not None else None
            ),
            "latency": {
                "predict_only": predict_summary,
                "decode_and_preprocess": _latency_summary(preprocessing_ms),
                "host_to_device_and_predict": accelerator_summary,
                "end_to_end": pipeline_summary,
            },
            "throughput_images_per_second": 1_000 / accelerator_summary["mean_ms"],
            "predict_only_images_per_second": 1_000 / predict_summary["mean_ms"],
            "end_to_end_images_per_second": 1_000 / pipeline_summary["mean_ms"],
            "cuda_memory": pipeline_memory,
            "predict_only_cuda_memory": predict_memory,
            "environment": {
                "python_version": sys.version.split()[0],
                "platform": platform.platform(),
                "torch_version": str(torch.__version__),
                "cuda_version": torch.version.cuda,
                "cudnn_version": torch.backends.cudnn.version(),
                "cudnn_benchmark": torch.backends.cudnn.benchmark,
                "cudnn_deterministic": torch.backends.cudnn.deterministic,
                "matmul_allow_tf32": torch.backends.cuda.matmul.allow_tf32,
                "cudnn_allow_tf32": torch.backends.cudnn.allow_tf32,
                "float32_matmul_precision": torch.get_float32_matmul_precision(),
            },
            "energy_joules_per_image": None,
            "energy_note": "Not measured; latency alone is not an energy measurement",
        }
    finally:
        detector.train(was_training)

def _nvml_handle_for_cuda(nvml: Any, device: torch.device) -> tuple[Any, str]:
    """Match by CUDA UUID; never assume CUDA and NVML ordinals are identical."""
    properties = torch.cuda.get_device_properties(device)
    cuda_uuid = getattr(properties, "uuid", None)
    if cuda_uuid:
        uuid_text = cuda_uuid.decode("ascii") if isinstance(cuda_uuid, bytes) else str(cuda_uuid)
        if not uuid_text.startswith(("GPU-", "MIG-")):
            uuid_text = "GPU-" + uuid_text
        try:
            return nvml.nvmlDeviceGetHandleByUUID(uuid_text), "cuda_uuid"
        except Exception as error:
            raise RuntimeError(f"Cannot match the CUDA UUID to NVML: {error}") from error
    if nvml.nvmlDeviceGetCount() == 1:
        return nvml.nvmlDeviceGetHandleByIndex(0), "single_nvml_device"
    raise RuntimeError(
        "CUDA UUID is unavailable and multiple NVML GPUs exist; "
        "refusing to measure an ambiguous physical device"
    )


def _nvml_counter_energy_window(
    predict_once: Any, nvml: Any, handle: Any, duration_seconds: float,
) -> dict[str, Any]:
    """Counter units are mJ; inference failures propagate rather than being hidden."""
    method = "nvml_total_energy_counter"
    try:
        before = int(nvml.nvmlDeviceGetTotalEnergyConsumption(handle))
        if before < 0:
            raise ValueError("negative energy counter")
    except Exception as error:
        return {"status": "unavailable", "method": method, "reason": str(error)}
    start = time.perf_counter()
    count = 0
    while count == 0 or time.perf_counter() - start < duration_seconds:
        predict_once()
        count += 1
    try:
        after = int(nvml.nvmlDeviceGetTotalEnergyConsumption(handle))
    except Exception as error:
        return {"status": "unavailable", "method": method, "reason": str(error)}
    elapsed = time.perf_counter() - start
    if after <= before:
        return {
            "status": "unavailable", "method": method,
            "reason": "Energy counter did not advance or was reset during measurement",
        }
    joules = (after - before) / 1_000.0
    return {
        "status": "measured", "method": method,
        "inferences": count, "duration_seconds": elapsed,
        "gpu_energy_joules": joules, "energy_joules_per_image": joules / count,
        "average_gpu_power_watts": joules / elapsed,
        "counter_start_millijoules": before,
        "counter_end_millijoules": after,
        "power_samples": [],
    }


def _nvml_power_energy_window(
    predict_once: Any, nvml: Any, handle: Any,
    duration_seconds: float, sample_interval_seconds: float,
) -> dict[str, Any]:
    """Integrate measured GPU power, with explicit boundary samples and no TDP estimate."""
    method = "nvml_power_integration"

    def read_watts() -> float:
        watts = float(nvml.nvmlDeviceGetPowerUsage(handle)) / 1_000.0
        if not math.isfinite(watts) or watts <= 0:
            raise ValueError("NVML power reading must be positive and finite")
        return watts

    try:
        first_watts = read_watts()
    except Exception as error:
        return {"status": "unavailable", "method": method, "reason": str(error)}
    start = time.perf_counter()
    samples = [(start, first_watts)]
    errors = []
    stop = threading.Event()

    def sample_power() -> None:
        while not stop.wait(sample_interval_seconds):
            try:
                watts = read_watts()
                samples.append((time.perf_counter(), watts))
            except Exception as error:
                errors.append(str(error))
                return

    worker = threading.Thread(target=sample_power, daemon=True)
    worker.start()
    count = 0
    try:
        while count == 0 or time.perf_counter() - start < duration_seconds:
            predict_once()
            count += 1
            if errors:
                break
    finally:
        stop.set()
        worker.join()
    if errors:
        return {"status": "unavailable", "method": method, "reason": errors[0]}
    try:
        final_watts = read_watts()
    except Exception as error:
        return {"status": "unavailable", "method": method, "reason": str(error)}
    samples.append((time.perf_counter(), final_watts))
    elapsed = samples[-1][0] - start
    # Two endpoints alone are not adequate for the sampled-power fallback.
    if len(samples) < 3:
        return {
            "status": "unavailable", "method": method,
            "reason": "Too few power samples; increase the measurement window",
        }
    joules = sum(
        (right[0] - left[0]) * (left[1] + right[1]) * 0.5
        for left, right in zip(samples, samples[1:])
    )
    return {
        "status": "measured", "method": method,
        "inferences": count, "duration_seconds": elapsed,
        "gpu_energy_joules": joules, "energy_joules_per_image": joules / count,
        "average_gpu_power_watts": joules / elapsed,
        "power_sample_interval_seconds_requested": sample_interval_seconds,
        "power_sample_count": len(samples),
        "power_samples": [
            {"elapsed_seconds": timestamp - start, "power_watts": watts}
            for timestamp, watts in samples
        ],
    }


@torch.inference_mode()
def measure_anomaly_inference_energy(
    detector: Any, loader: Any, *, device: torch.device | str,
    repeats: int = 3, duration_seconds: float = 30.0,
    warmup_seconds: float = 5.0, sample_interval_seconds: float = 0.1,
    method: str = "auto",
) -> dict[str, Any]:
    """Measure device-wide GPU J/image in a separate sustained predict-only phase.

    The model and one preprocessed input are already GPU resident. Each complete
    predict is synchronized, and its output discarded. Includes GPU idle/boundary
    energy, excludes CPU, I/O and host-to-device energy. Raw total energy is not
    idle-subtracted and is not attributed to an individual process. Sensor errors
    yield an explicit unavailable/partial report; prediction errors propagate.
    """
    if not isinstance(repeats, int) or isinstance(repeats, bool) or repeats < 1:
        raise ValueError("Energy repeats must be a positive integer")
    for name, value in (
        ("duration_seconds", duration_seconds),
        ("sample_interval_seconds", sample_interval_seconds),
    ):
        if not math.isfinite(value) or value <= 0:
            raise ValueError(f"{name} must be positive and finite")
    if not math.isfinite(warmup_seconds) or warmup_seconds < 0:
        raise ValueError("warmup_seconds must be non-negative and finite")
    if method not in {"auto", "counter", "power"}:
        raise ValueError("Energy method must be auto, counter or power")
    if getattr(loader, "batch_size", None) != 1 or getattr(loader, "num_workers", 0) != 0:
        raise ValueError("Energy requires batch_size=1 and num_workers=0")
    if not detector.is_fitted:
        raise RuntimeError("Restore the detector before measuring energy")
    device = torch.device(device)
    report = {
        "schema_version": 1,
        "status": "unavailable", "reason": None, "method": None,
        "metric": "device_wide_gpu_energy_per_inference",
        "scope": (
            "predict-only repeated synchronized inference; model and one preprocessed "
            "image are CUDA resident; excludes input transfer, CPU and loader work"
        ),
        "device": str(device), "batch_size": 1,
        "repeats_requested": repeats,
        "duration_seconds_per_repeat_requested": duration_seconds,
        "warmup_seconds_requested": warmup_seconds,
        "sample_interval_seconds_requested": sample_interval_seconds,
        "method_requested": method,
        "idle_subtracted": False, "process_attributed": False,
        "energy_joules_per_image": None,
        "average_gpu_power_watts": None,
        "energy_joules_per_image_std": None,
        "total_inferences": 0, "runs": [], "warnings": [],
    }
    if device.type != "cuda" or not torch.cuda.is_available():
        report["reason"] = "CUDA GPU is required for NVML energy measurements"
        return report
    try:
        import pynvml
    except ImportError:
        report["reason"] = "Install nvidia-ml-py (import pynvml) to read GPU energy/power"
        return report
    try:
        pynvml.nvmlInit()
    except Exception as error:
        report["reason"] = f"NVML initialization failed: {error}"
        return report

    was_training = detector.training
    try:
        try:
            handle, mapping = _nvml_handle_for_cuda(pynvml, device)
            report["device_mapping"] = mapping
            uuid = pynvml.nvmlDeviceGetUUID(handle)
            report["nvml_gpu_uuid"] = uuid.decode() if isinstance(uuid, bytes) else str(uuid)
            name = pynvml.nvmlDeviceGetName(handle)
            report["gpu_name"] = name.decode() if isinstance(name, bytes) else str(name)
        except Exception as error:
            report["reason"] = f"NVML device mapping failed: {error}"
            return report
        try:
            processes = pynvml.nvmlDeviceGetComputeRunningProcesses(handle)
            report["gpu_compute_process_count_at_start"] = len(processes)
            if len(processes) > 1:
                report["warnings"].append(
                    "Multiple compute processes are visible on this GPU; "
                    "device-wide energy cannot be assigned exclusively to this model"
                )
        except Exception:
            report["gpu_compute_process_count_at_start"] = None
        try:
            report["power_limit_watts"] = pynvml.nvmlDeviceGetPowerManagementLimit(handle) / 1_000.0
        except Exception:
            report["power_limit_watts"] = None
        detector.to(device).eval()
        try:
            batch = next(iter(loader))
        except StopIteration as error:
            raise ValueError("Energy measurement loader is empty") from error
        images = batch["image"].to(device)
        del batch
        if images.ndim != 4 or images.shape[0] != 1 or images.dtype != torch.float32:
            raise ValueError("Energy comparison requires one FP32 BCHW image")
        report["input_shape"] = list(images.shape)
        report["input_dtype"] = str(images.dtype)

        def predict_once() -> None:
            prediction = detector.predict(images)
            torch.cuda.synchronize(device)
            del prediction

        torch.cuda.synchronize(device)
        warmup_start = time.perf_counter()
        predict_once()
        while time.perf_counter() - warmup_start < warmup_seconds:
            predict_once()
        report["warmup_seconds_actual"] = time.perf_counter() - warmup_start
        chosen_method = "power" if method == "power" else "counter"
        for repeat in range(repeats):
            if chosen_method == "counter":
                run = _nvml_counter_energy_window(
                    predict_once, pynvml, handle, duration_seconds
                )
                # Auto chooses one successful backend on the first repeat only;
                # never average counter and integrated-power repeats together.
                if repeat == 0 and method == "auto" and run["status"] != "measured":
                    report["warnings"].append("Counter unavailable: " + str(run.get("reason")))
                    chosen_method = "power"
                    run = _nvml_power_energy_window(
                        predict_once, pynvml, handle,
                        duration_seconds, sample_interval_seconds,
                    )
            else:
                run = _nvml_power_energy_window(
                    predict_once, pynvml, handle,
                    duration_seconds, sample_interval_seconds,
                )
            run["repeat"] = repeat + 1
            report["runs"].append(run)
            if run["status"] != "measured":
                report["status"] = "partial" if repeat else "unavailable"
                report["reason"] = run.get("reason", "GPU energy sensor failed")
                return report
        runs = report["runs"]
        total_images = sum(run["inferences"] for run in runs)
        total_joules = sum(run["gpu_energy_joules"] for run in runs)
        total_seconds = sum(run["duration_seconds"] for run in runs)
        values = [run["energy_joules_per_image"] for run in runs]
        mean = sum(values) / len(values)
        report.update({
            "status": "measured", "reason": None,
            "method": runs[0]["method"],
            "total_inferences": total_images,
            "total_gpu_energy_joules": total_joules,
            "total_duration_seconds": total_seconds,
            "energy_joules_per_image": total_joules / total_images,
            "average_gpu_power_watts": total_joules / total_seconds,
            "energy_joules_per_image_std": (
                math.sqrt(sum((value - mean) ** 2 for value in values) / (len(values) - 1))
                if len(values) > 1 else 0.0
            ),
        })
        return report
    finally:
        detector.train(was_training)
        pynvml.nvmlShutdown()


In [ ]:
from __future__ import annotations

import torch



ESSENTIAL_METRIC_NAMES = (
    "image_auroc",
    "image_average_precision",
    "pixel_auroc",
    "pixel_average_precision",
)


class ExactBinaryMetrics:
    """Exact binary AUROC and average precision for small score collections."""

    def __init__(self) -> None:
        self._scores: list[torch.Tensor] = []
        self._targets: list[torch.Tensor] = []

    @torch.no_grad()
    def update(self, scores: torch.Tensor, targets: torch.Tensor) -> None:
        scores = scores.detach().reshape(-1).cpu()
        targets = targets.detach().reshape(-1).cpu().bool()
        if scores.numel() != targets.numel():
            raise ValueError("scores and targets must contain the same number of values")
        if not scores.numel():
            return
        if not scores.is_floating_point() or not torch.isfinite(scores).all():
            raise ValueError("scores must contain finite floating-point values")
        self._scores.append(scores.clone())
        self._targets.append(targets.clone())

    def compute(self) -> dict[str, float]:
        if not self._scores:
            raise ValueError("AUROC and average precision require both target classes")
        scores = torch.cat(self._scores)
        targets = torch.cat(self._targets)
        positives = int(targets.sum())
        negatives = targets.numel() - positives
        if positives == 0 or negatives == 0:
            raise ValueError("AUROC and average precision require both target classes")

        order = torch.argsort(scores, descending=True)
        sorted_scores = scores[order]
        sorted_targets = targets[order]
        _, group_counts = torch.unique_consecutive(sorted_scores, return_counts=True)
        group_ends = group_counts.cumsum(0) - 1
        true_positives = sorted_targets.cumsum(0)[group_ends].double()
        false_positives = (~sorted_targets).cumsum(0)[group_ends].double()
        recall = true_positives / positives
        false_positive_rate = false_positives / negatives
        precision = true_positives / (true_positives + false_positives)

        zero = torch.zeros(1, dtype=torch.float64)
        auroc = torch.trapezoid(
            torch.cat((zero, recall)),
            torch.cat((zero, false_positive_rate)),
        )
        recall_increment = recall - torch.cat((zero, recall[:-1]))
        average_precision = (precision * recall_increment).sum()
        return {
            "auroc": float(auroc),
            "average_precision": float(average_precision),
        }


class BinaryHistogramMetrics:
    """Memory-bounded binary AUROC and average precision accumulator."""

    def __init__(self, num_bins: int = 2048) -> None:
        if num_bins < 2:
            raise ValueError("num_bins must be at least 2")
        self.num_bins = int(num_bins)
        self.positive_histogram = torch.zeros(num_bins, dtype=torch.int64)
        self.negative_histogram = torch.zeros(num_bins, dtype=torch.int64)

    @torch.no_grad()
    def update(
        self,
        scores: torch.Tensor,
        targets: torch.Tensor,
        valid_mask: torch.Tensor | None = None,
    ) -> None:
        scores = scores.detach().reshape(-1).cpu()
        targets = targets.detach().reshape(-1).cpu().bool()
        if scores.numel() != targets.numel():
            raise ValueError("scores and targets must contain the same number of values")
        if valid_mask is not None:
            valid_mask = valid_mask.detach().reshape(-1).cpu().bool()
            if valid_mask.numel() != scores.numel():
                raise ValueError("valid_mask must match scores")
            scores = scores[valid_mask]
            targets = targets[valid_mask]
        if not scores.numel():
            return
        if not scores.is_floating_point() or not torch.isfinite(scores).all():
            raise ValueError("scores must contain finite floating-point values")
        if scores.min() < 0 or scores.max() > 1:
            raise ValueError("scores must be normalized to [0, 1]")

        bins = (scores * self.num_bins).long().clamp(max=self.num_bins - 1)
        self.positive_histogram += torch.bincount(
            bins[targets], minlength=self.num_bins
        )
        self.negative_histogram += torch.bincount(
            bins[~targets], minlength=self.num_bins
        )

    def compute(self) -> dict[str, float]:
        positives = int(self.positive_histogram.sum())
        negatives = int(self.negative_histogram.sum())
        if positives == 0 or negatives == 0:
            raise ValueError("AUROC and average precision require both target classes")

        true_positives = self.positive_histogram.flip(0).cumsum(0).double()
        false_positives = self.negative_histogram.flip(0).cumsum(0).double()
        recall = true_positives / positives
        false_positive_rate = false_positives / negatives
        precision = true_positives / (true_positives + false_positives).clamp_min(1)

        zero = torch.zeros(1, dtype=torch.float64)
        auroc = torch.trapezoid(
            torch.cat((zero, recall)),
            torch.cat((zero, false_positive_rate)),
        )
        recall_increment = recall - torch.cat((zero, recall[:-1]))
        average_precision = (precision * recall_increment).sum()
        return {
            "auroc": float(auroc),
            "average_precision": float(average_precision),
        }


class AnomalyMetrics:
    """Accumulate primary, target-wheel, and score-saturation metrics."""

    def __init__(self, histogram_bins: int = 2048) -> None:
        self.image = ExactBinaryMetrics()
        self.pixel = BinaryHistogramMetrics(histogram_bins)
        self.target_wheel_pixel = BinaryHistogramMetrics(histogram_bins)
        self.num_images = 0
        self.image_scores_at_zero = 0
        self.image_scores_at_one = 0

    @torch.no_grad()
    def update(
        self,
        prediction: AnomalyPrediction,
        labels: torch.Tensor,
        anomaly_masks: torch.Tensor,
        *,
        image_scores: torch.Tensor | None = None,
        valid_pixel_mask: torch.Tensor | None = None,
        target_wheel_mask: torch.Tensor | None = None,
    ) -> None:
        labels = labels.reshape(-1)
        if labels.shape != prediction.anomaly_score.shape:
            raise ValueError("labels must match anomaly_score shape")
        metric_image_scores = (
            prediction.anomaly_score if image_scores is None else image_scores
        ).reshape(-1)
        if metric_image_scores.shape != labels.shape:
            raise ValueError("image_scores must match labels")

        anomaly_masks = anomaly_masks > 0
        if anomaly_masks.ndim == 3:
            anomaly_masks = anomaly_masks.unsqueeze(1)
        if anomaly_masks.shape != prediction.anomaly_map.shape:
            raise ValueError("anomaly_masks must match anomaly_map shape")

        def prepare_mask(mask: torch.Tensor | None, name: str) -> torch.Tensor | None:
            if mask is None:
                return None
            mask = mask > 0
            if mask.ndim == 3:
                mask = mask.unsqueeze(1)
            if mask.shape != prediction.anomaly_map.shape:
                raise ValueError(f"{name} must match anomaly_map shape")
            return mask

        valid_pixel_mask = prepare_mask(valid_pixel_mask, "valid_pixel_mask")
        target_wheel_mask = prepare_mask(target_wheel_mask, "target_wheel_mask")
        if target_wheel_mask is None:
            target_wheel_mask = torch.ones_like(anomaly_masks, dtype=torch.bool)

        normalized_scores = prediction.anomaly_score.detach()
        self.num_images += normalized_scores.numel()
        self.image_scores_at_zero += int((normalized_scores == 0).sum())
        self.image_scores_at_one += int((normalized_scores == 1).sum())

        self.image.update(metric_image_scores, labels)
        self.pixel.update(
            prediction.anomaly_map,
            anomaly_masks,
            valid_mask=valid_pixel_mask,
        )
        self.target_wheel_pixel.update(
            prediction.anomaly_map,
            anomaly_masks,
            valid_mask=target_wheel_mask,
        )

    def compute(self) -> dict[str, float | int]:
        image = self.image.compute()
        pixel = self.pixel.compute()
        target_wheel_pixel = self.target_wheel_pixel.compute()
        saturated = self.image_scores_at_zero + self.image_scores_at_one
        return {
            "image_auroc": image["auroc"],
            "image_average_precision": image["average_precision"],
            "pixel_auroc": pixel["auroc"],
            "pixel_average_precision": pixel["average_precision"],
            "target_wheel_pixel_auroc": target_wheel_pixel["auroc"],
            "target_wheel_pixel_average_precision": (
                target_wheel_pixel["average_precision"]
            ),
            "normalized_image_scores_at_zero": self.image_scores_at_zero,
            "normalized_image_scores_at_one": self.image_scores_at_one,
            "normalized_image_score_saturation_fraction": (
                saturated / self.num_images if self.num_images else 0.0
            ),
        }




def update_metrics_from_batch(
    metrics: AnomalyMetrics,
    prediction: AnomalyPrediction,
    batch: dict[str, torch.Tensor],
    *,
    image_scores: torch.Tensor | None = None,
    restrict_pixels_to_target_mask: bool = False,
) -> None:
    """Update full-frame metrics plus the additional target-wheel view."""
    valid_pixel_mask = batch["target_mask"] if restrict_pixels_to_target_mask else None
    metrics.update(
        prediction,
        batch["label"],
        batch["anomaly_mask"],
        image_scores=image_scores,
        valid_pixel_mask=valid_pixel_mask,
        target_wheel_mask=batch["target_mask"],
    )


In [ ]:
from __future__ import annotations

import csv
import json
from collections import deque
from collections.abc import Mapping, Sequence
from pathlib import Path
from typing import Any

import torch



DEFAULT_GROUP_FIELDS = ("severity", "camera_pose", "lighting", "wear")


def _score_statistics(values: list[float]) -> dict[str, float | int]:
    if not values:
        return {"count": 0}
    scores = torch.tensor(values, dtype=torch.float64)
    return {
        "count": scores.numel(),
        "minimum": float(scores.min()),
        "q25": float(torch.quantile(scores, 0.25)),
        "median": float(torch.quantile(scores, 0.50)),
        "q75": float(torch.quantile(scores, 0.75)),
        "maximum": float(scores.max()),
        "mean": float(scores.mean()),
        "std": float(scores.std(unbiased=False)),
    }


def _connected_components(mask: torch.Tensor) -> list[list[int]]:
    """Return 4-connected positive regions as flattened pixel indices."""
    mask = mask.detach().cpu().bool().squeeze()
    if mask.ndim != 2:
        raise ValueError("PRO masks must be two-dimensional after squeezing")
    height, width = mask.shape
    remaining = set(torch.nonzero(mask.flatten(), as_tuple=False).flatten().tolist())
    components: list[list[int]] = []
    while remaining:
        start = remaining.pop()
        queue = deque([start])
        component = [start]
        while queue:
            index = queue.popleft()
            row, column = divmod(index, width)
            neighbours = []
            if row > 0:
                neighbours.append(index - width)
            if row + 1 < height:
                neighbours.append(index + width)
            if column > 0:
                neighbours.append(index - 1)
            if column + 1 < width:
                neighbours.append(index + 1)
            for neighbour in neighbours:
                if neighbour in remaining:
                    remaining.remove(neighbour)
                    queue.append(neighbour)
                    component.append(neighbour)
        components.append(component)
    return components


class PerRegionOverlap:
    """Memory-bounded PRO curve and normalized AUPRO accumulator."""

    def __init__(self, num_bins: int = 256, max_fpr: float = 0.30) -> None:
        if num_bins < 2:
            raise ValueError("num_bins must be at least 2")
        if not 0 < max_fpr <= 1:
            raise ValueError("max_fpr must be in (0, 1]")
        self.num_bins = int(num_bins)
        self.max_fpr = float(max_fpr)
        self.normal_histogram = torch.zeros(num_bins, dtype=torch.int64)
        self.region_overlap_sum = torch.zeros(num_bins, dtype=torch.float64)
        self.num_regions = 0

    @torch.no_grad()
    def update(
        self,
        scores: torch.Tensor,
        anomaly_masks: torch.Tensor,
        valid_mask: torch.Tensor | None = None,
    ) -> None:
        scores = scores.detach().cpu()
        anomaly_masks = anomaly_masks.detach().cpu() > 0
        if scores.ndim == 3:
            scores = scores.unsqueeze(1)
        if anomaly_masks.ndim == 3:
            anomaly_masks = anomaly_masks.unsqueeze(1)
        if scores.shape != anomaly_masks.shape:
            raise ValueError("scores and anomaly_masks must have matching shapes")
        if valid_mask is None:
            valid_mask = torch.ones_like(anomaly_masks, dtype=torch.bool)
        else:
            valid_mask = valid_mask.detach().cpu() > 0
            if valid_mask.ndim == 3:
                valid_mask = valid_mask.unsqueeze(1)
            if valid_mask.shape != scores.shape:
                raise ValueError("valid_mask must match scores")

        for score, anomaly_mask, valid in zip(scores, anomaly_masks, valid_mask):
            score = score.squeeze(0)
            anomaly_mask = anomaly_mask.squeeze(0) & valid.squeeze(0)
            valid = valid.squeeze(0)
            normal_scores = score[valid & ~anomaly_mask]
            if normal_scores.numel():
                bins = (normal_scores * self.num_bins).long().clamp(
                    min=0, max=self.num_bins - 1
                )
                self.normal_histogram += torch.bincount(
                    bins, minlength=self.num_bins
                )
            flattened_scores = score.flatten()
            for component in _connected_components(anomaly_mask):
                component_scores = flattened_scores[component]
                bins = (component_scores * self.num_bins).long().clamp(
                    min=0, max=self.num_bins - 1
                )
                histogram = torch.bincount(bins, minlength=self.num_bins)
                self.region_overlap_sum += (
                    histogram.flip(0).cumsum(0).double() / len(component)
                )
                self.num_regions += 1

    @staticmethod
    def _normalized_area(
        fpr: torch.Tensor,
        pro: torch.Tensor,
        max_fpr: float,
    ) -> float:
        below = fpr <= max_fpr
        clipped_fpr = fpr[below]
        clipped_pro = pro[below]
        if clipped_fpr[-1] < max_fpr:
            upper_index = int(torch.nonzero(fpr > max_fpr)[0])
            lower_index = upper_index - 1
            denominator = fpr[upper_index] - fpr[lower_index]
            weight = (
                0.0
                if denominator == 0
                else float((max_fpr - fpr[lower_index]) / denominator)
            )
            interpolated = pro[lower_index] + weight * (
                pro[upper_index] - pro[lower_index]
            )
            clipped_fpr = torch.cat(
                (clipped_fpr, torch.tensor([max_fpr], dtype=torch.float64))
            )
            clipped_pro = torch.cat((clipped_pro, interpolated.reshape(1)))
        area = torch.trapezoid(clipped_pro, clipped_fpr) / max_fpr
        return float(area)

    def compute(self) -> dict[str, Any]:
        total_normal = int(self.normal_histogram.sum())
        if total_normal == 0 or self.num_regions == 0:
            raise ValueError("PRO requires normal pixels and at least one anomaly region")
        fpr = self.normal_histogram.flip(0).cumsum(0).double() / total_normal
        pro = self.region_overlap_sum / self.num_regions
        fpr = torch.cat((torch.zeros(1, dtype=torch.float64), fpr))
        pro = torch.cat((torch.zeros(1, dtype=torch.float64), pro))

        reported_max_fprs = sorted({0.05, 0.10, self.max_fpr})
        aupro_by_max_fpr = {
            f"{max_fpr:.2f}": self._normalized_area(fpr, pro, max_fpr)
            for max_fpr in reported_max_fprs
        }
        return {
            "aupro": aupro_by_max_fpr[f"{self.max_fpr:.2f}"],
            "max_fpr": self.max_fpr,
            "aupro_by_max_fpr": aupro_by_max_fpr,
            "num_regions": self.num_regions,
            "curve": [
                {"false_positive_rate": float(x), "pro": float(y)}
                for x, y in zip(fpr, pro)
            ],
        }


class AnomalyDiagnostics:
    """Collect score distributions, grouped image/pixel metrics, and PRO."""

    def __init__(
        self,
        manifest_rows: Sequence[Mapping[str, str]],
        *,
        preprocessor: WheelPreprocessor,
        histogram_bins: int = 2048,
        pro_bins: int = 256,
        pro_max_fpr: float = 0.30,
        group_fields: Sequence[str] = DEFAULT_GROUP_FIELDS,
        num_extreme_examples: int = 6,
        restrict_pixels_to_target_mask: bool = False,
    ) -> None:
        if num_extreme_examples < 1:
            raise ValueError("num_extreme_examples must be at least 1")
        self.preprocessor = preprocessor
        self.histogram_bins = int(histogram_bins)
        self.group_fields = tuple(group_fields)
        self.num_extreme_examples = int(num_extreme_examples)
        self.restrict_pixels_to_target_mask = bool(restrict_pixels_to_target_mask)
        self.pro = PerRegionOverlap(pro_bins, pro_max_fpr)
        self.records: list[dict[str, Any]] = []
        self.group_metrics: dict[str, dict[str, BinaryHistogramMetrics]] = {
            field: {} for field in self.group_fields
        }
        self.group_image_counts: dict[str, dict[str, int]] = {
            field: {} for field in self.group_fields
        }
        self.top_clean: list[dict[str, Any]] = []
        self.bottom_hole: list[dict[str, Any]] = []
        self.anomalous_pixels = 0
        self.valid_pixels = 0
        self.pair_groups = {
            row.get("pair_id", ""): {
                field: row.get(field, "") for field in self.group_fields
            }
            for row in manifest_rows
            if row.get("condition") == "hole" and row.get("pair_id")
        }

    @staticmethod
    def _batch_value(metadata: Mapping[str, Any], key: str, index: int) -> Any:
        values = metadata.get(key)
        if values is None:
            return ""
        value = values[index]
        return value.item() if isinstance(value, torch.Tensor) else value

    def _group_value(
        self, metadata: Mapping[str, Any], field: str, index: int
    ) -> str:
        value = str(self._batch_value(metadata, field, index) or "")
        if value:
            return value
        pair_id = str(self._batch_value(metadata, "pair_id", index) or "")
        return str(self.pair_groups.get(pair_id, {}).get(field, ""))

    def _example(
        self,
        batch: Mapping[str, Any],
        prediction: AnomalyPrediction,
        raw_scores: torch.Tensor,
        index: int,
    ) -> dict[str, Any]:
        metadata = batch.get("metadata", {})
        return {
            "image": self.preprocessor.image_for_display(batch["image"][index]).cpu(),
            "target_mask": batch["target_mask"][index].cpu(),
            "anomaly_mask": batch["anomaly_mask"][index].cpu(),
            "anomaly_map": prediction.anomaly_map[index].detach().cpu(),
            "anomaly_score": float(prediction.anomaly_score[index].detach().cpu()),
            "raw_anomaly_score": float(raw_scores[index].detach().cpu()),
            "label": int(batch["label"][index]),
            "image_id": self._batch_value(metadata, "image_id", index),
        }

    @torch.no_grad()
    def update(
        self,
        prediction: AnomalyPrediction,
        raw_image_scores: torch.Tensor,
        batch: Mapping[str, Any],
    ) -> None:
        raw_image_scores = raw_image_scores.detach().reshape(-1).cpu()
        labels = torch.as_tensor(batch["label"]).reshape(-1).cpu()
        if raw_image_scores.shape != labels.shape:
            raise ValueError("raw_image_scores must match batch labels")
        anomaly_masks = batch["anomaly_mask"] > 0
        valid_mask = (
            batch["target_mask"] > 0
            if self.restrict_pixels_to_target_mask
            else torch.ones_like(anomaly_masks, dtype=torch.bool)
        )
        self.anomalous_pixels += int((anomaly_masks & valid_mask).sum())
        self.valid_pixels += int(valid_mask.sum())
        self.pro.update(prediction.anomaly_map, anomaly_masks, valid_mask)
        metadata = batch.get("metadata", {})

        for index, label_tensor in enumerate(labels):
            label = int(label_tensor)
            condition = "hole" if label else "clean"
            record = {
                "image_id": self._batch_value(metadata, "image_id", index),
                "pair_id": self._batch_value(metadata, "pair_id", index),
                "condition": condition,
                "raw_image_score": float(raw_image_scores[index]),
                "normalized_image_score": float(prediction.anomaly_score[index].cpu()),
            }
            for field in self.group_fields:
                value = self._group_value(metadata, field, index)
                record[field] = value
                if not value:
                    continue
                accumulator = self.group_metrics[field].setdefault(
                    value, BinaryHistogramMetrics(self.histogram_bins)
                )
                accumulator.update(
                    prediction.anomaly_map[index],
                    anomaly_masks[index],
                    valid_mask=valid_mask[index],
                )
                counts = self.group_image_counts[field]
                counts[value] = counts.get(value, 0) + 1
            self.records.append(record)

            example = self._example(batch, prediction, raw_image_scores, index)
            if label == 0:
                self.top_clean.append(example)
                self.top_clean.sort(
                    key=lambda item: item["raw_anomaly_score"], reverse=True
                )
                del self.top_clean[self.num_extreme_examples :]
            else:
                self.bottom_hole.append(example)
                self.bottom_hole.sort(key=lambda item: item["raw_anomaly_score"])
                del self.bottom_hole[self.num_extreme_examples :]

    def compute(self) -> dict[str, Any]:
        if not self.records or self.valid_pixels == 0:
            raise ValueError("Diagnostics require at least one evaluated image")
        grouped: dict[str, dict[str, Any]] = {}
        for field, values in self.group_metrics.items():
            grouped[field] = {}
            for value, metric in sorted(values.items()):
                positives = int(metric.positive_histogram.sum())
                negatives = int(metric.negative_histogram.sum())
                result = metric.compute() if positives and negatives else None
                grouped[field][value] = {
                    "pixel_average_precision": (
                        None if result is None else result["average_precision"]
                    ),
                    "anomalous_pixel_fraction": (
                        positives / (positives + negatives)
                        if positives + negatives
                        else 0.0
                    ),
                    "num_images": self.group_image_counts[field][value],
                }
        image_grouped: dict[str, dict[str, Any]] = {}
        for field in self.group_fields:
            image_grouped[field] = {}
            values = sorted(
                {str(row[field]) for row in self.records if row[field]}
            )
            for value in values:
                records = [row for row in self.records if row[field] == value]
                labels = torch.tensor(
                    [row["condition"] == "hole" for row in records],
                    dtype=torch.bool,
                )
                positives = int(labels.sum())
                negatives = labels.numel() - positives
                metrics = None
                if positives and negatives:
                    accumulator = ExactBinaryMetrics()
                    accumulator.update(
                        torch.tensor(
                            [row["raw_image_score"] for row in records],
                            dtype=torch.float32,
                        ),
                        labels,
                    )
                    metrics = accumulator.compute()
                image_grouped[field][value] = {
                    "image_auroc": None if metrics is None else metrics["auroc"],
                    "image_average_precision": (
                        None if metrics is None else metrics["average_precision"]
                    ),
                    "num_images": len(records),
                    "num_clean_images": negatives,
                    "num_anomalous_images": positives,
                }

        raw_scores = {
            condition: [
                row["raw_image_score"]
                for row in self.records
                if row["condition"] == condition
            ]
            for condition in ("clean", "hole")
        }
        return {
            "raw_image_score_distribution": {
                condition: _score_statistics(values)
                for condition, values in raw_scores.items()
            },
            "image_metrics_by_group": image_grouped,
            "pixel_average_precision_by_group": grouped,
            "anomalous_pixel_fraction": self.anomalous_pixels / self.valid_pixels,
            "random_pixel_ap_baseline": self.anomalous_pixels / self.valid_pixels,
            "pro": self.pro.compute(),
        }


In [ ]:
from __future__ import annotations

@torch.no_grad()
def evaluate_anomaly_detector(
    detector: AnomalyDetector,
    loader: Iterable[Mapping[str, Any]],
    *,
    device: torch.device | str,
    histogram_bins: int = 2048,
    restrict_pixels_to_target_mask: bool = False,
    description: str = "Evaluation",
    diagnostics: AnomalyDiagnostics | None = None,
) -> dict[str, float | int]:
    """Evaluate one fitted detector and optionally collect diagnostics."""
    if not detector.is_fitted:
        raise RuntimeError("The anomaly detector must be fitted before evaluation")
    device = torch.device(device)
    detector.to(device)
    detector.train(False)
    metrics = AnomalyMetrics(histogram_bins=histogram_bins)
    num_images = 0

    for batch in tqdm(loader, desc=description):
        images = batch["image"].to(device, non_blocking=True)
        predict_with_raw = getattr(detector, "predict_with_raw", None)
        if predict_with_raw is None:
            prediction = detector.predict(images)
            raw_image_scores = prediction.anomaly_score
        else:
            prediction, raw_image_scores, _ = predict_with_raw(images)
        update_metrics_from_batch(
            metrics,
            prediction,
            batch,
            image_scores=raw_image_scores,
            restrict_pixels_to_target_mask=restrict_pixels_to_target_mask,
        )
        if diagnostics is not None:
            diagnostics.update(prediction, raw_image_scores, batch)
        num_images += images.shape[0]
    if num_images == 0:
        raise ValueError("Evaluation loader produced no images")
    return metrics.compute()



def validate_sigma4_test_population(rows, manifest_path, source_manifest_sha256):
    """Require the original complete paired test population before inference."""
    if TEST_SPLIT != "test":
        raise ValueError("The sigma-4 rerun must use the held-out test split")
    if tuple(CAMERA_POSES) != ("A_overhead",):
        raise ValueError("The sigma-4 rerun must retain pose A only")
    actual_manifest = sha256_file(manifest_path)
    if actual_manifest != EXPECTED_MANIFEST_SHA256:
        raise ValueError("samples.csv differs from the original TG-512 dataset manifest")
    if source_manifest_sha256 is not None and actual_manifest != source_manifest_sha256:
        raise ValueError("Dataset manifest does not match the restored checkpoint")
    counts = Counter(row["condition"] for row in rows)
    if counts != Counter(EXPECTED_TEST_CONDITION_COUNTS):
        raise ValueError(f"Unexpected test population: {dict(counts)}")
    if len({row["image_id"] for row in rows}) != len(rows):
        raise ValueError("Duplicate image IDs in the selected test population")
    pairs = {}
    for row in rows:
        if row.get("split") != "test" or row.get("camera_pose") != "A_overhead":
            raise ValueError("The selected test population contains another split or pose")
        pair_id = row.get("pair_id")
        if not pair_id:
            raise ValueError("Every pose-A test sample must belong to a clean/hole pair")
        pairs.setdefault(pair_id, []).append(row)
        if any(not row.get(field) for field in ("lighting", "wear")):
            raise ValueError("Lighting and wear metadata are required for subgroup metrics")
        if row["condition"] == "hole" and row.get("severity") not in {"small", "medium", "large"}:
            raise ValueError("Hole severity metadata is missing or invalid")
    for pair_id, pair_rows in pairs.items():
        if len(pair_rows) != 2 or {row["condition"] for row in pair_rows} != {"clean", "hole"}:
            raise ValueError(f"Incomplete clean/hole test pair: {pair_id}")
        for field in ("camera_pose", "lighting", "wear"):
            if len({row[field] for row in pair_rows}) != 1:
                raise ValueError(f"Pair {pair_id} has inconsistent {field} metadata")
    return {"condition_counts": dict(counts), "num_images": len(rows),
            "num_pairs": len(pairs), "manifest_sha256": actual_manifest}


## Measurement protocol and validity checks

Use a fresh session with only this detector resident on the GPU. The frozen ResNet-18 backbone, discriminator and output postprocessing run in FP32; The selected profile's smoothing (sigma 1 for TG-384, sigma 4 for TG-512) is included in every `predict` call. No feature cache, autocast, compilation, quantization or synthetic-input shortcut is used.

The benchmark matches the dedicated PatchCore protocol: batch 1, 10 warm-up iterations and 50 timed iterations **per scope**, CUDA synchronization, sequential loader, no worker prefetch and no pinned-memory shortcut.

- **Predict-only:** the same preprocessed image is already on the device. This excludes loading, transforms and host-to-device transfer.
- **Host-to-device + predict:** one image is transferred after each loader iteration.
- **End-to-end:** the existing synthetic loader also reads and transforms target/anomaly masks. This is an evaluation pipeline, **not** a mask-free deployment measurement.
- Report CUDA allocated/reserved baseline, total peak and incremental peak separately. These are PyTorch allocator measurements, not process/driver memory.
- Latencies from the separate phases cannot be subtracted to infer transfer cost. Compare models on the same GPU and the same named scope.

The checkpoint's saved architecture, preprocessing and sigma are checked against the selected profile before restore; folder names alone are not reliable. TG-384 keeps its original sigma 1, while TG-512 applies sigma 4 only after restoring its saved configuration. This couples the measurement to each model's reported accuracy rather than changing smoothing for the benchmark. A manifest mismatch or unexpected validation count is recorded and warned about, not filtered: these do not affect the fixed input shape. There is no full-dataset mask audit or visibility gate. Energy is measured in the separate phase below; latency alone is not an energy estimate.

### GPU energy (separate from the latency windows)

The default is three **30 s** sustained predict-only windows after a **5 s**
warm-up, batch 1, FP32, and the same preprocessed image already on the GPU.
The complete predict call and its postprocessing run every iteration; outputs
are discarded. No feature cache, refit, recalibration or input shortcut is used.

NVML's total-energy counter is preferred (mJ converted to J). When unavailable
or non-advancing, the first window chooses the measured-power fallback:
100 ms requested sampling and trapezoidal integration. A single backend is
used for all successful repeats; a partial sensor failure leaves the aggregate
null and records the reason. Inference errors still propagate.

The result is **device-wide GPU J/image**, including GPU idle power and memory,
not per-process or whole-system energy. CPU, loader and input-transfer work are
excluded; idle power is not subtracted. The CUDA/NVML devices are matched by
UUID, and additional visible compute processes are flagged. Use a fresh T4
session with no other model resident for the comparison. Actual power and
sensor readings are saved, never estimated from nominal GPU power or latency.

The energy phase adds about 95 seconds when the counter succeeds. Read the
**final saved JSON or the post-energy printout**, not the earlier latency
printout whose energy field may still be null.


## Restore and measure the selected model

Run cells in order in a fresh T4 session. The default TG-384 profile restores
the final 384 × 384/layer2/sigma1 model and measures validation efficiency.
The later TG-512 sigma-4 test cells are skipped by default. No training,
calibration, model selection or checkpoint overwrite is performed.


In [ ]:
from __future__ import annotations

random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if REQUIRE_CUDA and DEVICE.type != "cuda":
    raise RuntimeError("Select a Kaggle GPU session, preferably Tesla T4 to match the other benchmarks")
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("highest")
if DEVICE.type == "cuda":
    gpu_name = torch.cuda.get_device_name(DEVICE)
    print(f"GPU: {gpu_name}")
    if "T4" not in gpu_name:
        print("WARNING: this is not a T4; do not compare these latencies directly with T4 results")

if CHECKPOINT_SOURCE == "google_drive":
    checkpoint_path, download_info = download_selected_checkpoint_from_drive()
else:
    if MODEL_CHECKPOINT_PATH is None:
        raise ValueError("Set MODEL_CHECKPOINT_PATH to your trusted final TinyGLASS model.ckpt")
    checkpoint_path = Path(MODEL_CHECKPOINT_PATH).expanduser()
    download_info = {"source": "local", "path": str(checkpoint_path)}
detector, restore_info = load_frozen_tinyglass(checkpoint_path, device=DEVICE)
print({
    "reference_profile": REFERENCE_PROFILE,
    "source_run_id": restore_info["source_run_id"],
    "input_size": detector.input_size,
    "feature_grid_resolution": detector.feature_grid_resolution,
    "saved_sigma": restore_info["saved_gaussian_sigma"],
    "inference_sigma": detector.gaussian_sigma,
    "checkpoint_mib": restore_info["checkpoint_size_bytes"] / 2**20,
    "training_performed": False,
})


In [ ]:
from __future__ import annotations

DATASET_ROOT = (
    Path(DATASET_ROOT_OVERRIDE).expanduser().resolve()
    if DATASET_ROOT_OVERRIDE is not None else find_dataset_root()
)
preprocessor = WheelPreprocessor(PreprocessingConfig(
    resize=MODEL_INPUT_SIZE,
    resize_shorter_side=None,
    center_crop=None,
    pose_crop_enabled=True,
    pose_crops=POSE_CROPS,
    normalize_mean=IMAGENET_MEAN,
    normalize_std=IMAGENET_STD,
    augmentations_enabled=False,
))
manifest_sha256 = sha256_file(DATASET_ROOT / "samples.csv")
source_manifest = restore_info["source_manifest_sha256"]
dataset_matches_source_manifest = None if source_manifest is None else manifest_sha256 == source_manifest
if dataset_matches_source_manifest is False:
    print("WARNING: attached samples.csv differs from the checkpoint dataset manifest; this will be recorded")

if BENCHMARK_ENABLED:
    benchmark_loader = build_dataloader(
        DATASET_ROOT, BENCHMARK_SPLIT,
        batch_size=BENCHMARK_BATCH_SIZE,
        num_workers=BENCHMARK_NUM_WORKERS,
        pin_memory=BENCHMARK_PIN_MEMORY,
        seed=RANDOM_SEED,
        preprocessing=preprocessor,
        camera_poses=CAMERA_POSES,
    )
    condition_counts = Counter(row["condition"] for row in benchmark_loader.dataset.rows)
    if condition_counts != Counter({"clean": 296, "hole": 96}):
        print(f"WARNING: validation counts differ from the original pose-A dataset: {dict(condition_counts)}")
    print({
        "dataset_root": str(DATASET_ROOT),
        "split": BENCHMARK_SPLIT,
        "camera_poses": CAMERA_POSES,
        "condition_counts": dict(condition_counts),
        "input_size": MODEL_INPUT_SIZE,
        "pose_crop": POSE_CROPS["A_overhead"],
        "manifest_sha256": manifest_sha256,
        "dataset_matches_source_manifest": dataset_matches_source_manifest,
    })

else:
    print("Efficiency benchmark disabled; the optional TG-512 test runs when enabled.")


In [ ]:
from __future__ import annotations

if BENCHMARK_ENABLED:
    print(f"Benchmarking {REFERENCE_PROFILE}: {MODEL_INPUT_SIZE}, "
          f"{EXPECTED_FEATURE_GRID_RESOLUTION}, sigma {INFERENCE_GAUSSIAN_SIGMA:g}")
    # Outputs of predict() are discarded inside the benchmark to avoid inflated peaks.
    benchmark_report = benchmark_anomaly_inference(
        detector, benchmark_loader, device=DEVICE,
        warmup_runs=BENCHMARK_WARMUP_RUNS,
        timed_runs=BENCHMARK_TIMED_RUNS,
        split=BENCHMARK_SPLIT,
    )
    if benchmark_report["input_shape"] != [1, 3, *MODEL_INPUT_SIZE]:
        raise ValueError("Unexpected benchmark input shape")
    if benchmark_report["input_dtype"] != "torch.float32":
        raise ValueError("This comparison requires FP32 inputs")
    benchmark_report.update({
        **restore_info,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "checkpoint_source": CHECKPOINT_SOURCE,
        "checkpoint_download": download_info,
        "camera_poses": list(CAMERA_POSES),
        "pose_crops": {pose: list(crop) for pose, crop in POSE_CROPS.items()},
        "preprocessing": {
            "resize": list(MODEL_INPUT_SIZE),
            "resize_shorter_side": None,
            "center_crop": None,
            "pose_crop_enabled": True,
            "normalize_mean": list(IMAGENET_MEAN),
            "normalize_std": list(IMAGENET_STD),
            "augmentations_enabled": False,
        },
        "dataset_root": str(DATASET_ROOT),
        "validation_condition_counts": dict(condition_counts),
        "manifest_sha256": manifest_sha256,
        "dataset_matches_source_manifest": dataset_matches_source_manifest,
        "pin_memory": BENCHMARK_PIN_MEMORY,
        "num_workers": BENCHMARK_NUM_WORKERS,
    })
    print(json.dumps(benchmark_report, indent=2, allow_nan=False))


In [ ]:
from __future__ import annotations

if BENCHMARK_ENABLED:
    # Defaults also allow this cell to run in an already-fitted live session,
    # without rerunning Configuration and changing the original run/output identity.
    ENERGY_ENABLED = globals().get("ENERGY_ENABLED", True)
    ENERGY_METHOD = globals().get("ENERGY_METHOD", "auto")
    ENERGY_REPEATS = globals().get("ENERGY_REPEATS", 3)
    ENERGY_DURATION_SECONDS = globals().get("ENERGY_DURATION_SECONDS", 30.0)
    ENERGY_WARMUP_SECONDS = globals().get("ENERGY_WARMUP_SECONDS", 5.0)
    ENERGY_POWER_SAMPLE_INTERVAL_SECONDS = globals().get("ENERGY_POWER_SAMPLE_INTERVAL_SECONDS", 0.1)

    if ENERGY_ENABLED:
        print(
            f"Measuring GPU energy: {ENERGY_REPEATS} x {ENERGY_DURATION_SECONDS:g} s "
            f"after {ENERGY_WARMUP_SECONDS:g} s warm-up; predict-only, batch 1, FP32"
        )
        energy_report = measure_anomaly_inference_energy(
            detector, benchmark_loader, device=DEVICE,
            repeats=ENERGY_REPEATS,
            duration_seconds=ENERGY_DURATION_SECONDS,
            warmup_seconds=ENERGY_WARMUP_SECONDS,
            sample_interval_seconds=ENERGY_POWER_SAMPLE_INTERVAL_SECONDS,
            method=ENERGY_METHOD,
        )
    else:
        energy_report = {
            "schema_version": 1, "status": "disabled",
            "reason": "ENERGY_ENABLED=False", "method": None,
            "energy_joules_per_image": None,
            "average_gpu_power_watts": None,
            "energy_joules_per_image_std": None,
            "runs": [], "warnings": [],
        }
    energy_report.update({
        "reference_profile": REFERENCE_PROFILE,
        "input_size": list(MODEL_INPUT_SIZE),
        "feature_grid_resolution": EXPECTED_FEATURE_GRID_RESOLUTION,
        "gaussian_sigma": float(detector.gaussian_sigma),
        "source_run_id": restore_info["source_run_id"],
        "checkpoint_path": restore_info["checkpoint_path"],
        "manifest_sha256": manifest_sha256,
        "camera_poses": list(CAMERA_POSES),
        "pose_crops": {pose: list(crop) for pose, crop in POSE_CROPS.items()},
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "training_performed": False,
        "calibration_performed": False,
    })
    benchmark_report["gpu_energy_measurement"] = energy_report
    benchmark_report["energy_joules_per_image"] = energy_report["energy_joules_per_image"]
    benchmark_report["energy_note"] = (
        "Measured device-wide GPU energy per predict-only inference via "
        + str(energy_report["method"])
        + "; excludes CPU/input transfer/loader; idle is not subtracted"
        if energy_report["status"] == "measured"
        else "GPU energy not measured completely: " + str(energy_report.get("reason"))
    )
    print({
        "energy_status": energy_report["status"],
        "energy_method": energy_report["method"],
        "gpu_energy_joules_per_image": energy_report["energy_joules_per_image"],
        "average_gpu_power_watts": energy_report["average_gpu_power_watts"],
        "energy_joules_per_image_std": energy_report["energy_joules_per_image_std"],
        "warnings": energy_report["warnings"],
        "reason": energy_report.get("reason"),
    })
    if energy_report["status"] not in {"measured", "disabled"}:
        print("WARNING: energy could not be measured reliably; inspect inference_energy_batch1.json")

    print("Updated benchmark JSON (includes the energy phase):")
    print(json.dumps(benchmark_report, indent=2, allow_nan=False))


In [ ]:
from __future__ import annotations

if BENCHMARK_ENABLED:
    benchmark_run_id = (
        f"tinyglass_{MODEL_INPUT_SIZE[0]}x{MODEL_INPUT_SIZE[1]}_"
        f"{EXPECTED_FEATURE_GRID_RESOLUTION}_A_crop720_"
        f"sigma{INFERENCE_GAUSSIAN_SIGMA:g}_efficiency__"
        + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
    )
    OUTPUT_DIR = OUTPUT_ROOT / benchmark_run_id
    benchmark_report["benchmark_run_id"] = benchmark_run_id
    energy_report["benchmark_run_id"] = benchmark_run_id
    report_path = save_json_atomic(benchmark_report, OUTPUT_DIR / "inference_benchmark_batch1.json")
    energy_path = save_json_atomic(energy_report, OUTPUT_DIR / "inference_energy_batch1.json")
    memory = benchmark_report["cuda_memory"] or {}
    predict_memory = benchmark_report["predict_only_cuda_memory"] or {}
    summary_row = {
        "benchmark_run_id": benchmark_run_id,
        "source_run_id": benchmark_report["source_run_id"],
        "reference_profile": REFERENCE_PROFILE,
        "feature_grid_resolution": EXPECTED_FEATURE_GRID_RESOLUTION,
        "model": benchmark_report["model"],
        "gpu": benchmark_report["gpu_name"],
        "input_height": MODEL_INPUT_SIZE[0],
        "input_width": MODEL_INPUT_SIZE[1],
        "batch_size": benchmark_report["batch_size"],
        "gaussian_sigma": benchmark_report["gaussian_sigma"],
        "parameters": benchmark_report["total_parameters"],
        "parameters_mib": benchmark_report["parameters_mib"],
        "buffers_mib": benchmark_report["buffers_mib"],
        "checkpoint_mib": benchmark_report["checkpoint_size_bytes"] / 2**20,
        "predict_only_mean_ms": benchmark_report["latency"]["predict_only"]["mean_ms"],
        "predict_only_p95_ms": benchmark_report["latency"]["predict_only"]["p95_ms"],
        "host_to_device_and_predict_mean_ms": benchmark_report["latency"]["host_to_device_and_predict"]["mean_ms"],
        "end_to_end_mean_ms": benchmark_report["latency"]["end_to_end"]["mean_ms"],
        "predict_only_images_per_second": benchmark_report["predict_only_images_per_second"],
        "host_to_device_and_predict_images_per_second": benchmark_report["throughput_images_per_second"],
        "end_to_end_images_per_second": benchmark_report["end_to_end_images_per_second"],
        "cuda_peak_allocated_mib": memory.get("peak_allocated_mib"),
        "cuda_peak_reserved_mib": memory.get("peak_reserved_mib"),
        "cuda_incremental_peak_allocated_mib": memory.get("incremental_peak_allocated_mib"),
        "predict_only_cuda_peak_allocated_mib": predict_memory.get("peak_allocated_mib"),
        "energy_joules_per_image": benchmark_report["energy_joules_per_image"],
        "energy_status": energy_report["status"],
        "energy_method": energy_report["method"],
        "average_gpu_power_watts": energy_report["average_gpu_power_watts"],
        "energy_joules_per_image_std": energy_report["energy_joules_per_image_std"],
    }
    summary_path = OUTPUT_DIR / "inference_benchmark_batch1.csv"
    with summary_path.open("w", encoding="utf-8", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=list(summary_row))
        writer.writeheader()
        writer.writerow(summary_row)
    print(f"JSON report: {report_path}")
    print(f"CSV summary: {summary_path}")
    print(f"Energy report: {energy_path}")
    print("Efficiency reports saved; the separate test section runs when enabled.")


## Optional TG-512 held-out test with sigma 4

Skipped by default. To run it, set REFERENCE_PROFILE="TG-512",
TEST_EVALUATION_ENABLED=True and BENCHMARK_ENABLED=False in Configuration,
then run all cells in a fresh session.

The manifest SHA-256, pose, split, 404 clean/404 hole counts and complete pairing
are checked before inference. A mismatch stops this reference evaluation rather
than silently changing its population. No sample is filtered by mask visibility,
severity, score or image appearance. Clean samples inherit their paired hole's
severity through the authoritative diagnostic accumulator.

Inference uses the same normalized fixed crop at 512 × 512, FP32, layer3 and the
raw maximum patch score. Sigma 4 affects localization maps, while the image score
is computed before smoothing. Its prior selection is fixed: this is a single
test evaluation, not a sigma sweep. Reference image metrics are compared only as
an informational consistency check, without selecting a checkpoint or threshold.

Pixel histograms use 2048 bins and PRO uses 256, matching the original contract.
The primary pixel metrics cover the entire processed image; target-wheel metrics
are additional audits. Diagnostic histograms and a bounded set of examples avoid
retaining all dense anomaly maps. CPU diagnostic work is part of evaluation time
and is not an inference-efficiency measurement. Outputs are placed in a new
timestamped directory, and the checkpoint/manifest hashes are checked again after
inference to detect changes during the evaluation.


In [ ]:
from __future__ import annotations

if TEST_EVALUATION_ENABLED:
    if REFERENCE_PROFILE != "TG-512":
        raise ValueError("Select REFERENCE_PROFILE='TG-512' for the optional sigma-4 test")
    if float(INFERENCE_GAUSSIAN_SIGMA) != 4.0 or float(detector.gaussian_sigma) != 4.0:
        raise ValueError("This test rerun requires the selected sigma 4")
    if tuple(detector.input_size) != (512, 512) or detector.feature_grid_resolution != "layer3":
        raise ValueError("The reference test requires TG-512 with layer3 features")
    test_loader = build_dataloader(
        DATASET_ROOT, TEST_SPLIT, batch_size=TEST_BATCH_SIZE,
        num_workers=TEST_NUM_WORKERS, pin_memory=TEST_PIN_MEMORY,
        seed=RANDOM_SEED, preprocessing=preprocessor, camera_poses=CAMERA_POSES,
    )
    test_population = validate_sigma4_test_population(
        test_loader.dataset.rows, DATASET_ROOT / "samples.csv",
        restore_info["source_manifest_sha256"],
    )
    checkpoint_sha256_before_test = sha256_file(checkpoint_path)
    test_diagnostics = AnomalyDiagnostics(
        test_loader.dataset.rows, preprocessor=preprocessor,
        histogram_bins=TEST_HISTOGRAM_BINS, pro_bins=TEST_PRO_BINS,
        pro_max_fpr=TEST_PRO_MAX_FPR, group_fields=TEST_GROUP_FIELDS,
        num_extreme_examples=TEST_NUM_EXTREME_EXAMPLES,
        restrict_pixels_to_target_mask=False,
    )
    test_run_id = (
        "tinyglass_512x512_layer3_A_crop720_sigma4_test__"
        + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
    )
    TEST_OUTPUT_DIR = OUTPUT_ROOT / "test_sigma4" / test_run_id
    TEST_OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
    print({**test_population, "gaussian_sigma": detector.gaussian_sigma,
           "input_size": detector.input_size, "output_dir": str(TEST_OUTPUT_DIR)})


In [ ]:
from __future__ import annotations

if TEST_EVALUATION_ENABLED:
    test_started = time.perf_counter()
    with torch.inference_mode(), torch.autocast(device_type=DEVICE.type, enabled=False):
        test_metrics = evaluate_anomaly_detector(
            detector, test_loader, device=DEVICE, histogram_bins=TEST_HISTOGRAM_BINS,
            restrict_pixels_to_target_mask=False,
            description="TG-512 held-out test (sigma 4)", diagnostics=test_diagnostics,
        )
    test_duration_seconds = time.perf_counter() - test_started
    if len(test_diagnostics.records) != test_population["num_images"]:
        raise RuntimeError("The evaluation did not process the complete test population")
    checkpoint_sha256_after_test = sha256_file(checkpoint_path)
    if checkpoint_sha256_after_test != checkpoint_sha256_before_test:
        raise RuntimeError("The source checkpoint changed during test evaluation")
    if sha256_file(DATASET_ROOT / "samples.csv") != test_population["manifest_sha256"]:
        raise RuntimeError("The dataset manifest changed during test evaluation")
    metrics_path = save_json_atomic({"test": test_metrics}, TEST_OUTPUT_DIR / "metrics.json")
    reference_image_metric_differences = {
        name: float(test_metrics[name]) - reference
        for name, reference in REFERENCE_TEST_IMAGE_METRICS.items()
    }
    print("Test metrics at sigma 4:")
    print(json.dumps(test_metrics, indent=2, allow_nan=False))
    print({"reference_image_metric_differences": reference_image_metric_differences,
           "test_duration_seconds": test_duration_seconds, "metrics_path": str(metrics_path)})


In [ ]:
from __future__ import annotations

if TEST_EVALUATION_ENABLED:
    diagnostic_result = test_diagnostics.compute()
    diagnostic_summary = {
        **diagnostic_result,
        "pro": {key: value for key, value in diagnostic_result["pro"].items() if key != "curve"},
    }
    diagnostics_path = save_json_atomic(
        diagnostic_summary, TEST_OUTPUT_DIR / "test_diagnostics.json",
    )
    scores_path = TEST_OUTPUT_DIR / "test_image_scores.csv"
    with scores_path.open("w", encoding="utf-8", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=list(test_diagnostics.records[0]))
        writer.writeheader()
        writer.writerows(test_diagnostics.records)
    pro_path = TEST_OUTPUT_DIR / "test_pro_curve.csv"
    with pro_path.open("w", encoding="utf-8", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=["false_positive_rate", "pro"])
        writer.writeheader()
        writer.writerows(diagnostic_result["pro"]["curve"])

    subgroup_rows = []
    for field in TEST_GROUP_FIELDS:
        for group, image_metrics in diagnostic_summary["image_metrics_by_group"][field].items():
            pixel_metrics = diagnostic_summary["pixel_average_precision_by_group"][field][group]
            if image_metrics["num_images"] != pixel_metrics["num_images"]:
                raise RuntimeError(f"Image/pixel subgroup count mismatch: {field}/{group}")
            subgroup_rows.append({
                "group_field": field, "group": group, **image_metrics,
                "pixel_average_precision": pixel_metrics["pixel_average_precision"],
                "anomalous_pixel_fraction": pixel_metrics["anomalous_pixel_fraction"],
                "gaussian_sigma": float(detector.gaussian_sigma),
            })
    subgroup_path = TEST_OUTPUT_DIR / "test_subgroup_metrics.csv"
    with subgroup_path.open("w", encoding="utf-8", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=list(subgroup_rows[0]))
        writer.writeheader()
        writer.writerows(subgroup_rows)
    artifacts = [metrics_path, diagnostics_path, scores_path, pro_path, subgroup_path]
    test_provenance = {
        **restore_info,
        "test_run_id": test_run_id, "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "checkpoint_source": CHECKPOINT_SOURCE, "checkpoint_download": download_info,
        "checkpoint_sha256": checkpoint_sha256_after_test,
        "checkpoint_preserved": True,
        "dataset_root": str(DATASET_ROOT), **test_population,
        "split": TEST_SPLIT, "camera_poses": list(CAMERA_POSES),
        "preprocessing": {
            "resize": list(MODEL_INPUT_SIZE), "resize_shorter_side": None,
            "center_crop": None, "pose_crop_enabled": True,
            "pose_crops": {pose: list(crop) for pose, crop in POSE_CROPS.items()},
            "normalize_mean": list(IMAGENET_MEAN), "normalize_std": list(IMAGENET_STD),
            "augmentations_enabled": False,
        },
        "histogram_bins": TEST_HISTOGRAM_BINS, "pro_bins": TEST_PRO_BINS,
        "pro_max_fpr": TEST_PRO_MAX_FPR, "group_fields": list(TEST_GROUP_FIELDS),
        "restrict_pixels_to_target_mask": False, "input_dtype": "torch.float32",
        "batch_size": TEST_BATCH_SIZE, "num_workers": TEST_NUM_WORKERS,
        "pin_memory": TEST_PIN_MEMORY, "random_seed": RANDOM_SEED,
        "device": str(DEVICE),
        "gpu_name": torch.cuda.get_device_name(DEVICE) if DEVICE.type == "cuda" else None,
        "python_version": platform.python_version(), "torch_version": str(torch.__version__),
        "test_duration_seconds": test_duration_seconds,
        "accuracy_evaluation_performed": True, "training_performed": False,
        "calibration_performed": False, "model_selection_performed": False,
        "sigma_selection_performed": False, "test_used_for_selection": False,
        "reference_test_image_metrics": REFERENCE_TEST_IMAGE_METRICS,
        "reference_image_metric_differences": reference_image_metric_differences,
        "artifacts": {path.name: {"size_bytes": path.stat().st_size,
                                  "sha256": sha256_file(path)} for path in artifacts},
    }
    provenance_path = save_json_atomic(
        test_provenance, TEST_OUTPUT_DIR / "test_evaluation_provenance.json",
    )
    print("Subgroup metrics at sigma 4 (scores printed as %):")
    print("Group field | Group | Clean | Hole | Image AUROC | Image AP | Pixel AP")
    for row in subgroup_rows:
        percentages = [
            "--" if row[name] is None else f"{100 * row[name]:.2f}"
            for name in ("image_auroc", "image_average_precision", "pixel_average_precision")
        ]
        print(f"{row['group_field']} | {row['group']} | {row['num_clean_images']} | "
              f"{row['num_anomalous_images']} | " + " | ".join(percentages))
    print("AUPRO:", diagnostic_summary["pro"]["aupro_by_max_fpr"])
    print(f"Download the complete test output folder: {TEST_OUTPUT_DIR}")


## Saved outputs

The default TG-384 workflow creates a new timestamped directory such as
/kaggle/working/tinyglass_efficiency/tinyglass_384x384_layer2_A_crop720_sigma1_efficiency__.../
containing:

- inference_benchmark_batch1.json: latency for all three scopes, throughput,
  parameters, checkpoint size, CUDA memory, final energy result and provenance.
- inference_benchmark_batch1.csv: one summary row, including reference profile,
  input size, feature layer, sigma and GPU J/image.
- inference_energy_batch1.json: individual energy windows, power, variability,
  sensor backend and any reason that prevents a complete measurement.

Download this complete folder from Kaggle for the TG-384 efficiency comparison.
The GPU energy phase takes about 95 seconds; use the saved report after it ends.

With REFERENCE_PROFILE="TG-512" and TEST_EVALUATION_ENABLED=True, the optional
test workflow instead creates a separate directory under
/kaggle/working/tinyglass_efficiency/test_sigma4/ with metrics.json,
test_diagnostics.json, test_subgroup_metrics.csv, test_image_scores.csv,
test_pro_curve.csv and test_evaluation_provenance.json. JSON/CSV scores are
fractions; printed subgroup scores are percentages. The benchmark uses
validation, while accuracy evaluation uses held-out test.
